# Insurance Policy RAG — evaluation notebook v3.2 (Colab T4)

Runs the RAG system on **your chunk file** and evaluates it on **test set v3** (113 questions keyed to the PDF:
25 dev, 88 held-out test; 19 not answerable). System: BGE-small dense search + BM25 (premium tables as a separate
index, searched only for cost questions) → **bge-reranker-v2-m3** → context (1,500 tokens) → relevance gate →
**Qwen3-8B, 4-bit, thinking on** in a LangGraph pipeline. Answers are scored by deterministic checks plus
**HHEM-2.1-Open** (is each sentence supported by the context?). No LLM judge, no RAGAS, no tracing service.

**Thinking without cut-offs.** The model thinks for at most `think_budget` tokens (default 1,024). If it has not
finished by then, it is told to stop thinking and answer (Qwen3's own budget method), and the answer gets its own
1,000 tokens. The thinking is kept separately (in `results.csv`) and never mixed into the answer. Decoding uses Qwen's
recommended sampling for thinking mode, seeded from the prompt, so the same question gives the same answer every run.

**Run it**
1. *Runtime → Change runtime type → T4 GPU.*
2. Check cell 1 (the defaults are the baseline run). *Runtime → Run all.*
3. Cell 3 asks for the chunk file (`chunks_from_pdf_v27.json`). Results stay in this runtime (no Drive permission).
   During long runs, download `rag_eval/<run_name>/answer_cache.jsonl` from the Files panel now and then. After a
   disconnect, set `RESUME_FROM = "upload"` in cell 2 and choose that file (or a results zip): finished answers are
   not generated again. `USE_DRIVE = True` in cell 2 copies everything to Google Drive instead (asks for permission).
4. Time with thinking on: about 5–8 min of model downloads (~20 GB) on the first run, then about **45 min for the
   25 dev questions** and **about 3–3.5 hours for all 113**. Easiest: run `"splits": ["dev"]` first, then
   `["dev", "test"]` with the same `run_name` (the dev answers come from the cache). Keep the tab open.
5. A results zip downloads at the end (with Drive also under `insurance_rag_eval/<run_name>/`): `report.md`,
   `results.csv`, `scorecard.csv`, the run JSON, the answer cache, `labels_to_fill.csv`. It contains policy text:
   keep it out of the public repository.

**Later runs** — give each run its own `run_name`; answers are reused whenever the prompt and model are unchanged.
| Goal | Set in cell 1 |
|---|---|
| test a change against a baseline | `overrides` (a CONFIG value, e.g. `{"think_budget": 1536}`) or a model name, plus `compare_with` = the baseline zip |
| thinking off (fast, ~15 s per answer) | `enable_thinking = False` |
| another answer model on identical contexts | `contexts_from` and `compare_with` = the baseline zip, `llm_model` = the candidate |
| compare chunk files (retrieval only, minutes) | `mode = "retrieval"`, then each chunk file in turn |
| re-check saved answers after the checks change (no answer model, ~10 min) | `mode = "rescore"`, `rescore_from` = the run's zip |
| the 15 hard questions (not scored) | `hard_questions = True`, click into cell 9, *Runtime → Run before* |
| check the checks | fill `labels_to_fill.csv`, set `LABELS_FILE` in cell 13, re-run cell 13 |

**Decision rule (fixed in advance, test split only):** adopt a change only if *fully correct* improves significantly
(McNemar exact; Holm-adjusted when several candidates are compared), groundedness and correct abstention do not get
significantly worse (paired bootstrap), and the cost stays within 150 s median per answer (thinking on) and 14 GB
peak GPU memory.

## Setup

In [ ]:
# ================================================================
# 1) Settings: the only cell you normally edit
# ================================================================
SETTINGS = {
    # Input
    "chunk_file": None,           # path to your chunk file (.json or .csv); None = upload it when cell 3 runs
    # What to run
    "run_name": "qwen3_8b_think",  # names the results folder and zip (give every run its own name)
    "splits": ["dev", "test"],    # "dev" = 25 questions (~45 min with thinking) for debugging; "test" = 88 held-out questions
    "mode": "full",               # "full" = answers + all metrics; "retrieval" = no answer model (minutes), to compare chunk files;
                                  # "rescore" = re-check a saved run's answers with this notebook's checks and HHEM (~10 min)
    "rescore_from": None,         # mode "rescore": the saved run's results zip or JSON (or "upload")
    "overrides": {},              # optional CONFIG changes for an experiment, e.g. {"context_token_budget": 2600}
    "contexts_from": None,        # model comparison: a previous run's zip/json (or "upload"); its saved contexts are answered again
    "compare_with": None,         # baseline run's zip/json (or "upload"), or [baseline, other candidates]: paired tests + decision
    "groundedness": True,         # sentence-level check of every answer with HHEM-2.1-Open (~1 GB, a few minutes)
    "evidence_removed_test": True,  # 4 extra answers with the evidence taken out of the context
    "repeat_check": True,         # 1 answer regenerated without the cache (determinism)
    "hard_questions": False,      # cell 8b: 15 extra hard questions with expected answers (not scored; ~25 min)
    "save_to_drive": False,       # set by USE_DRIVE in cell 2 (Google Drive copy of the cache and results)
    # Models
    "embed_model": "BAAI/bge-small-en-v1.5",
    "reranker_model": "BAAI/bge-reranker-v2-m3",
    "llm_model": "Qwen/Qwen3-8B",
    "llm_4bit": True,             # NF4 4-bit: ~6.5 GB on a T4 (fp16 does not fit next to the other models)
    "enable_thinking": True,      # Qwen3 thinks before answering; thinking has its own budget (CONFIG["think_budget"])
    "sampling": True,             # Qwen's recommended sampling for the mode, seeded per prompt (repeatable); False = greedy
    "grounding_model": "vectara/hallucination_evaluation_model",   # HHEM-2.1-Open, Apache 2.0
}

# Pipeline parameters (v2.6 values; the budget is now counted in the answer model's tokens)
CONFIG = {
    "k_dense": 32, "k_bm25": 32, "rrf_k": 60,     # candidates from each search leg; RRF only orders their union
    "use_bm25": True,
    "bge_query_prefix": "Represent this sentence for searching relevant passages: ",
    "rerank_with_label": True,       # the reranker sees "label: text" (labels carry section and clause names)
    "rerank_max_length": 1024,
    "context_token_budget": 1500,    # hard cap for the context, labels included
    "title_pinning": True,           # sections whose heading words all appear in the question go first
    "neighbour_expansion": True,     # add the chunk a selected chunk continues into / depends on
    "seed_score_floor": -5.0,        # reranked chunks below this do not fill the context
    "rerank_score_threshold": -5.0,  # gate: refuse when the best reranker score is below this (re-check with the gate report)
    "premium_routing": True,         # premium-table chunks: a separate index, searched only for premium/cost questions
    "k_premium": 12,                 # candidates from each search leg of the premium index
    "think_budget": 1024,            # thinking tokens; then the model is told to stop thinking and answer
    "max_new_tokens": 1000,          # answer tokens (after the thinking)
    "oom_retry_context_tokens": 1000,
}

# Fixed before any results (agreed decision rule and budget)
DECISION = {
    "alpha": 0.05,                 # McNemar exact test on "fully correct", held-out test split
    "n_bootstrap": 10_000,         # paired bootstrap resamples of questions
    "max_median_seconds": 150,     # per generated answer (thinking on; 45 s was the limit without thinking)
    "max_peak_gpu_gb": 14.0,
    "hhem_threshold": 0.5,         # a sentence is supported when HHEM scores it at or above this
    "kappa_target": 0.7,           # agreement of the automatic verdict with human labels
}
CONFIG.update(SETTINGS["overrides"])            # experiment changes apply before anything uses CONFIG
NOTEBOOK_VERSION = "v3.2"
print(f"{NOTEBOOK_VERSION} | run '{SETTINGS['run_name']}' | splits {SETTINGS['splits']} | mode {SETTINGS['mode']} | "
      f"LLM {SETTINGS['llm_model']} ({'4-bit' if SETTINGS['llm_4bit'] else 'fp16'}, thinking "
      f"{'on, budget ' + str(CONFIG['think_budget']) if SETTINGS['enable_thinking'] else 'off'}, "
      f"{'seeded sampling' if SETTINGS['sampling'] else 'greedy'}) | reranker {SETTINGS['reranker_model']}")


In [ ]:
# ================================================================
# 2) Install, imports, output folder
# ================================================================
import os, re, sys, subprocess, importlib.util
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before torch touches the GPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

_PKGS = [("rank_bm25", "rank_bm25"), ("langgraph", "langgraph"), ("faiss", "faiss-cpu"),
         ("sentence_transformers", "sentence-transformers"), ("transformers", "transformers"), ("accelerate", "accelerate")]
try:
    import torch
    if torch.cuda.is_available() and SETTINGS["llm_4bit"]:
        _PKGS.append(("bitsandbytes", "bitsandbytes"))
except ImportError:
    _PKGS.append(("torch", "torch"))
_missing = [pip for mod, pip in _PKGS if importlib.util.find_spec(mod) is None]
try:                                         # Qwen3 needs transformers >= 4.51
    import importlib.metadata as _md0
    if tuple(int(x) for x in re.findall(r"\d+", _md0.version("transformers"))[:2]) < (4, 51):
        _missing.append("transformers>=4.51")
except Exception:
    pass
if _missing:
    print("Installing:", " ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_missing], check=True)

import re, io, json, time, math, gc, hashlib, datetime, zipfile, shutil, platform, statistics, traceback, random
import importlib.metadata as _md
from pathlib import Path
from typing import TypedDict, List
import numpy as np
import pandas as pd
import torch, faiss

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:          # no "google" package at all: not Colab
    IN_COLAB = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

OUT = Path("rag_eval") / SETTINGS["run_name"]
OUT.mkdir(parents=True, exist_ok=True)
RUN_LOG = []

def log(msg: str):
    """A problem worth knowing about; printed now and saved with the results."""
    print("  NOTE:", msg)
    RUN_LOG.append(msg)

# Storage: this runtime only, so no Google Drive permission is asked. Results stay in rag_eval/<run_name>/ and the
# results zip downloads at the end (cell 13). The answer cache can be downloaded at any time from the Files panel
# (left sidebar: rag_eval/<run_name>/answer_cache.jsonl). To continue an interrupted run, or to reuse the answers of
# an earlier run, set RESUME_FROM = "upload" and choose that answer_cache.jsonl or the run's results zip.
USE_DRIVE = False        # True = copy the cache and results to Google Drive after every answer (asks for permission)
RESUME_FROM = None       # None, "upload", or a path to an answer_cache.jsonl or results .zip from an earlier run
SETTINGS["save_to_drive"] = USE_DRIVE
DRIVE = None
if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE = Path("/content/drive/MyDrive/insurance_rag_eval") / SETTINGS["run_name"]
        DRIVE.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        log(f"Google Drive not available ({type(e).__name__}: {e}); results stay in this runtime")

def mirror(path: Path):
    if DRIVE is not None:
        try:
            shutil.copy(path, DRIVE / Path(path).name)
        except Exception as e:
            log(f"copy to Drive failed for {Path(path).name}: {e}")

def version(pkg: str):
    try:
        return _md.version(pkg)
    except Exception:
        return None

_TF = tuple(int(x) for x in re.findall(r"\d+", version("transformers") or "0.0")[:2])

def dtype_kw(dt) -> dict:
    """from_pretrained takes dtype= from transformers 4.56 on (torch_dtype= before)."""
    return {"dtype": dt} if _TF >= (4, 56) else {"torch_dtype": dt}

def gpu_gb():
    if not torch.cuda.is_available():
        return None
    free, total = torch.cuda.mem_get_info()
    return round((total - free) / 1e9, 2)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def upload_one(what: str) -> Path:
    """Colab file picker; returns the saved path."""
    if not IN_COLAB:
        raise FileNotFoundError(f"set a path for the {what} in SETTINGS (uploading works in Colab)")
    from google.colab import files
    print(f"Choose the {what}:")
    name, data = next(iter(files.upload().items()))
    p = Path("uploads") / name
    p.parent.mkdir(exist_ok=True)
    p.write_bytes(data)
    return p

if RESUME_FROM:              # earlier answers go into this run's cache; they are reused where prompt and model match
    _src = upload_one("answer cache (answer_cache.jsonl or the results .zip)") if RESUME_FROM == "upload" \
        else Path(RESUME_FROM)
    _raw = _src.read_bytes()
    if _src.suffix.lower() == ".zip":
        _names = [n for n in zipfile.ZipFile(io.BytesIO(_raw)).namelist() if n.endswith("answer_cache.jsonl")]
        _raw = zipfile.ZipFile(io.BytesIO(_raw)).read(_names[0]) if _names else b""
    _lines = [l for l in _raw.decode("utf-8").splitlines() if l.strip()]
    with open(OUT / "answer_cache.jsonl", "a", encoding="utf-8") as _fh:
        _fh.write("".join(l + "\n" for l in _lines))
    print(f"Resuming from {_src.name}: {len(_lines)} cached answers added (used where the prompt and model match)")
    if not _lines:
        log(f"no answer_cache.jsonl found in {_src.name}")

print(f"Colab: {IN_COLAB} | device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.mem_get_info()[1] / 1e9:.1f} GB)" if DEVICE == "cuda" else "") +
      f" | torch {torch.__version__} | transformers {version('transformers')} | results in {OUT}/"
      + (f" and Drive {DRIVE}/" if DRIVE else " (this runtime only: download the zip at the end)"))
if DEVICE == "cpu" and SETTINGS["mode"] == "full":
    log("no GPU: answering will take hours. In Colab choose Runtime > Change runtime type > T4 GPU.")


In [ ]:
# ================================================================
# 3) Load the chunk file (the direct input; no chunking happens in this notebook)
# ================================================================
# Accepts the PDF chunk file (chunks_from_pdf_v27.json/.csv) and the older chunks.json. Fields used:
#   text, section_id (required); section_title, label, embed_text, clause, index ("policy"/"premium"),
#   group (pieces that belong together), kind / is_table (optional; filled in when missing).
CHUNK_PATH = Path(SETTINGS["chunk_file"]) if SETTINGS["chunk_file"] else upload_one("chunk file (.json or .csv)")
_raw = CHUNK_PATH.read_bytes()
CHUNK_SHA = hashlib.sha256(_raw).hexdigest()
if CHUNK_PATH.suffix.lower() == ".csv":
    _rows = pd.read_csv(io.BytesIO(_raw), keep_default_na=False).to_dict("records")
else:
    _data = json.loads(_raw.decode("utf-8"))
    _rows = _data.get("chunks", _data) if isinstance(_data, dict) else _data

def _clean(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else str(v).strip()

final_chunks, _skipped = [], 0
for r in _rows:
    text, sid = _clean(r.get("text")), _clean(r.get("section_id"))
    if not text or not sid:
        _skipped += 1
        continue
    title = _clean(r.get("section_title"))
    clause = _clean(r.get("clause")) or None
    label = _clean(r.get("label")) or (f"Section {sid} - {title}" if title else f"Section {sid}")
    index = _clean(r.get("index")).lower() or "policy"
    final_chunks.append({
        "idx": len(final_chunks), "section_id": sid, "section_title": title, "clause": clause, "label": label,
        "index": "premium" if index == "premium" else "policy",
        "unit": _clean(r.get("group")) or sid,                 # neighbour expansion stays inside a unit
        "is_table": str(r.get("is_table")).lower() == "true" or _clean(r.get("kind")) == "table",
        "text": text, "embed_text": _clean(r.get("embed_text")) or f"{label}: {text}"})
assert final_chunks, f"no usable chunks in {CHUNK_PATH.name}: each needs 'text' and 'section_id'"

SECTIONS = {}                                  # section id -> title and text, in document order
for c in final_chunks:
    s = SECTIONS.setdefault(c["section_id"], {"section_id": c["section_id"], "section_title": c["section_title"],
                                              "index": c["index"], "text": []})
    s["text"].append(c["text"])
for s in SECTIONS.values():
    s["text"] = " ".join(s["text"])
N_PREMIUM = sum(c["index"] == "premium" for c in final_chunks)
print(f"Chunk file {CHUNK_PATH.name} (sha256 {CHUNK_SHA[:12]}…): {len(final_chunks)} chunks "
      f"({len(final_chunks) - N_PREMIUM} policy, {N_PREMIUM} premium-table), {len(SECTIONS)} sections"
      + (f"; {_skipped} rows skipped (no text or section_id)" if _skipped else ""))
if not N_PREMIUM:
    print("  No premium-table chunks: premium questions will be scored as missing evidence (expected for the old text file).")


## Test set

In [ ]:
# ================================================================
# 4) Test set v3 (frozen): 113 questions keyed to the PDF; dev = 25, held-out test = 88
# ================================================================
# Fields per question:
#   split       "dev" (debugging, the compact 24 + N12) or "test" (held out: adopt/reject decisions only)
#   sections    PDF section ids holding the answer (used for citations and the evidence-removed test)
#   evidence    evidence passages, each a list of alternative anchor phrases (PDF wording | old text-file
#               wording). A chunk is relevant when it contains one; this scores retrieval for any chunk file.
#   facts       key facts; each a list of alternative phrasings ("re:" = regular expression on normalised text)
#   checks      table_cells (plan-table values), polarity (yes/no), ordered (facts must appear in this order),
#               off_scope (sections of another cover), must_not (known false statements / wrong table rows),
#               derived_figures (figures the answer may compute), expect_admit (must say what is not shown),
#               absent_terms (not-answerable questions: terms that must not occur in the policy)
# Changing anything here makes a new version (the fingerprint below) and baselines must be re-run.
TESTSET = [
    {'id': 'G01', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What expenses are covered under the Overseas Travel Secure option? Give complete details.', 'sections': ['4.11'], 'evidence': [['minimum hospitalization of 5 consecutive days and has also'], ['tickets most basic economy class airfare in'], ['15 000 per day only'], ['maximum upto 30 days in a policy'], ['expenses such as meals laundry transport are not']], 'facts': [['5 consecutive days'], ['accompanying person'], ['economy'], ['15,000'], ['30 days'], ['meals, laundry', 'meals']], 'reference': 'Overseas Travel Secure (4.11), available only with the Global or Global Plus plan, pays economy airfare for the insured person and one accompanying person and accommodation for the accompanying person up to Rs. 15,000 per day for up to 30 days in a policy year, if the overseas doctor advised at least 5 consecutive days of hospitalization and a Global Health Cover claim was accepted; meals, laundry and other transport are not paid.'},
    {'id': 'G02', 'split': 'dev', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is the Plus Benefit? Explain in detail.', 'sections': ['4.4'], 'evidence': [['sum equal to 50 of the base'], ['accumulated up to 100 of base sum'], ['will be applied irrespective of number of'], ['is received within the grace period']], 'facts': [['50%'], ['100%'], ['irrespective', 'regardless', 're:\\b(even|whether) (if|or not|though|when)\\b[^.]{0,50}\\bclaim', 're:\\bno matter\\b', 're:\\bnot (be )?(reduced|affected|impacted|dependent|linked|lost)\\b[^.]{0,40}\\bclaim', 're:\\b(does|do) not depend on\\b[^.]{0,20}\\bclaim', 're:\\bstill\\b[^.]{0,30}\\b(get|receive|added|eligible|entitled|apply|applies|be added)\\b'], ['grace period']], 'must_not': ['plus benefit (is |are )?not (applicable|available|payable|extended)'], 'role': 'definition; known false claim', 'reference': 'On each renewal without a break, 50% of the expiring Base Sum Insured is added, accumulating up to 100% of the Base Sum Insured irrespective of claims; it needs renewal within the Grace Period and can be used only for Section 3 and Section 4.3 claims.'},
    {'id': 'G03', 'split': 'dev', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What are the exact inclusions for pre-hospitalization expenses?', 'sections': ['3.5'], 'evidence': [['for up to 60 days immediately prior to']], 'facts': [['60 days'], ['3.1', 'admissible hospitalization']], 'off_scope': ['4.9', '4.10', '3.7'], 'must_not': ['\\broom rent\\b|\\bboarding\\b|\\bsurgeon|\\banaesthetist|operation theatre|\\bintravenous\\b|\\bprosthetic'], 'role': 'single fact; rules of other covers', 'reference': 'Pre-hospitalization medical expenses are covered for up to 60 days immediately before an admissible hospitalization under Section 3.1.'},
    {'id': 'G04', 'split': 'dev', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What are the exact inclusions for post-hospitalization expenses?', 'sections': ['3.6'], 'evidence': [['for up to 180 days from the date']], 'facts': [['180 days'], ['discharge']], 'off_scope': ['4.9', '4.10', '3.1.1', '3.7'], 'must_not': ['\\broom rent\\b|\\bboarding\\b|\\bsurgeon|\\banaesthetist|operation theatre|\\bintravenous\\b|\\bprosthetic', 'pre hospitali[sz]ation (medical )?expenses (are|is) not (included|payable|covered)'], 'role': 'single fact; rules of other covers', 'reference': 'Post-hospitalization medical expenses are covered for up to 180 days from discharge, following an admissible hospitalization claim under Section 3.1.'},
    {'id': 'G05', 'split': 'dev', 'type': 'procedure', 'answerable': True, 'paraphrase': False, 'question': 'How to file reimbursement claims? Give me step wise complete procedure.', 'sections': ['11.6', '11.7'], 'evidence': [['within 30 days of date of', 'be submitted within 30 days from the date'], ['within 15 days from completion of', 'be submitted within 15 days from the completion'], ['duly completed claim form']], 'facts': [['30 days'], ['15 days'], ['claim form'], ['discharge']], 'off_scope': ['11.2', '11.3', '11.4'], 'role': 'procedure; cashless steps do not belong', 'reference': 'Submit the claim documents within 30 days of discharge (hospitalization, day care, pre-hospitalization) or within 15 days of finishing post-hospitalization treatment, with the claim form, photo ID, discharge summary, final bill and the other documents in Section 11.7.'},
    {'id': 'G06', 'split': 'dev', 'type': 'procedure', 'answerable': True, 'paraphrase': False, 'question': 'What is the procedure for cashless claims in India? Give me step wise complete procedure.', 'sections': ['11.2'], 'evidence': [['may be taken in a network provider and is subject to pre'], ['the time of discharge the insured person'], ['the company for reimbursement']], 'facts': [['network provider'], ['pre authorization', 'preauthorization'], ['discharge'], ['reimbursement']], 'off_scope': ['11.3', '4.9', '4.10'], 'role': 'procedure; overseas steps do not belong', 'reference': 'Treatment at a Network Provider subject to pre-authorization: the provider sends the cashless request, the Company issues a pre-authorization letter, the insured signs the discharge papers and pays non-medical expenses, and if cashless is denied the claim can be filed for reimbursement.'},
    {'id': 'G07', 'split': 'dev', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What are the exact specific conditions with a 24-month waiting period? Give complete details.', 'sections': ['10.1'], 'evidence': [['the expiry of 24 months of continuous coverage'], ['non infective arthritis'], ['cataract and other disorders'], ['hernia'], ['for claims arising due to an accident']], 'facts': [['24 months', '24 month'], ['non infective arthritis'], ['cataract'], ['hernia'], ['accident']], 'role': 'list that is in the context', 'reference': 'Under Excl02 the listed conditions (such as non-infective arthritis, gall bladder disease, kidney stones, cataract) and procedures (such as hernia repair, joint replacement) are excluded for 24 months of continuous coverage, except claims arising from an accident.'},
    {'id': 'G08', 'split': 'dev', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What are the standard exclusions under the my:Optima Secure policy? Give complete details.', 'sections': ['10.1'], 'evidence': [['excl01'], ['excl02'], ['excl03'], ['excl06'], ['excl09'], ['excl18']], 'facts': [['excl01', 'pre existing'], ['excl02', '24 months'], ['excl03', '30 day'], ['excl06', 'obesity'], ['excl09', 'hazardous'], ['excl18', 'maternity']], 'role': 'long list (context budget, answer length)', 'reference': 'Standard exclusions Excl01 to Excl18: pre-existing diseases (36 months), specified diseases (24 months), 30-day waiting period, investigation, rest cure, obesity surgery, gender change, cosmetic surgery, hazardous sports, breach of law, excluded providers, addiction, spas, supplements, refractive error, unproven treatment, infertility and maternity.'},
    {'id': 'G09', 'split': 'dev', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.', 'sections': ['2.1'], 'evidence': [['year 1 optima secure base sum insured', 'hospitalization claim year 1'], ['year 1 optima super secure base sum insured', 'hospitalization claim year 1'], ['year 2 optima secure base sum insured', 'hospitalization claim year 2'], ['year 3 optima secure base sum insured', 'hospitalization claim year 3']], 'facts': [['secure benefit'], ['20 lakh'], ['30 lakh'], ['40 lakh']], 'table_cells': [['secure', 'single', 1], ['secure', 'single', 2], ['secure', 'single', 3], ['secure', 'all', 1], ['secure', 'all', 2], ['secure', 'all', 3], ['super', 'single', 1], ['super', 'single', 2], ['super', 'single', 3], ['super', 'all', 1], ['super', 'all', 2], ['super', 'all', 3], ['secure', 'secure_benefit', 1], ['super', 'secure_benefit', 1]], 'must_not': ['neither of these benefits (are|is) available', '(optima )?secure (plan )?(does not|doesn t|do not) (offer|have|include|provide) (the |a |any )?secure benefit'], 'role': 'table: whole plan comparison, all years', 'reference': 'With a Rs. 10 lakh Base Sum Insured, the Secure Benefit is Rs. 10 lakh for Optima Secure and Rs. 20 lakh for Optima Super Secure. Maximum for a single claim: Optima Secure Rs. 20 lakh in Year 1, Rs. 25 lakh in Year 2, Rs. 30 lakh in Year 3; Optima Super Secure Rs. 30 lakh in Year 1, Rs. 35 lakh in Year 2, Rs. 40 lakh in Year 3. Maximum for all claims in a year: Optima Secure Rs. 30 lakh in Year 1, Rs. 35 lakh in Year 2, Rs. 40 lakh in Year 3; Optima Super Secure Rs. 40 lakh in Year 1, Rs. 45 lakh in Year 2, Rs. 50 lakh in Year 3.'},
    {'id': 'G10', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'What is the coverage for pet insurance under this policy?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['pet', 'pets', 'animal insurance'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'F01', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the age range of people covered under this policy?', 'sections': ['1'], 'evidence': [['the age group 91 days to 65 years']], 'facts': [['91 days'], ['65 years']], 'reference': 'The policy covers insured persons aged 91 days to 65 years.'},
    {'id': 'F02', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the maximum entry age for a dependent child?', 'sections': ['1'], 'evidence': [['entry age is 25 years']], 'facts': [['25 years', 're:\\b(?:aged?|ages|under|below|less than|younger than|up to|upto|until|till|before|over|above|at least|minimum of|maximum of|older than)(?: the)?(?: age)?(?: of)? 25\\b(?! (?:days|months|lakh))']], 'reference': 'A dependent child can enter up to age 25.'},
    {'id': 'F03', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How many adults and children can be covered in a family floater policy?', 'sections': ['1'], 'evidence': [['a maximum of 4 adults and a maximum']], 'facts': [['4 adults', 'four adults'], ['6 dependent children', 'six dependent children', '6 children']], 'reference': 'A family floater policy can include up to 4 adults and 6 dependent children.'},
    {'id': 'F04', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the minimum duration of hospitalization for a claim to be admissible?', 'sections': ['3.1.1'], 'evidence': [['minimum period of 24 consecutive hours only shall be'], ['all day care treatments']], 'facts': [['24 consecutive hours', '24 hours'], ['day care']], 'reference': 'Hospitalization must last at least 24 consecutive hours; the limit does not apply to day care treatment.'},
    {'id': 'F05', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the waiting period for pre-existing diseases?', 'sections': ['10.1'], 'evidence': [['the expiry of 36 months of continuous coverage']], 'facts': [['36 months']], 'reference': 'Pre-existing diseases are excluded until 36 months of continuous coverage (Excl01).'},
    {'id': 'F06', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the initial waiting period for illnesses after the policy starts?', 'sections': ['10.1'], 'evidence': [['any illness within 30 days from the first'], ['for claims arising due to an accident']], 'facts': [['30 days'], ['accident']], 'reference': 'Illness treated within 30 days of the first policy start is excluded, except claims arising from an accident (Excl03).'},
    {'id': 'F07', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the grace period for renewing the policy?', 'sections': ['12.9'], 'evidence': [['grace period of 30 days to maintain continuity'], ['coverage is not available during the grace period']], 'facts': [['30 days'], ['not available during the grace period', 'coverage is not available', 're:\\bno (?:coverage|cover|insurance cover)\\b[^.]{0,30}\\bgrace period', 're:\\b(?:not covered|no claims? (?:are |is )?(?:payable|admissible|paid))\\b[^.]{0,40}\\bgrace period', 're:\\bgrace period\\b[^.]{0,60}\\b(?:no (?:coverage|cover)|not covered|coverage (?:is |will )?not)']], 'reference': 'The policy can be renewed within a 30-day Grace Period without a break, but there is no coverage during the Grace Period.'},
    {'id': 'F08', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How long is the free look period?', 'sections': ['12.8'], 'evidence': [['look period of fifteen days from date of']], 'facts': [['fifteen days', '15 days']], 'reference': 'New individual policies have a free look period of fifteen days from receipt of the policy document.'},
    {'id': 'F09', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'Within how many days must the company settle a claim after receiving the last necessary document?', 'sections': ['12.3'], 'evidence': [['may be within 30 days from the date'], ['the claim within 45 days from the date']], 'facts': [['30 days'], ['45 days']], 'reference': 'Claims are settled or rejected within 30 days of the last necessary document, or within 45 days when an investigation is needed.'},
    {'id': 'F10', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the per claim deductible under the Global Health Cover?', 'sections': ['4.9', '4.10'], 'evidence': [['10 000 will apply separately for each and every claim except section']], 'facts': [['10,000']], 'reference': 'A per claim deductible of Rs. 10,000 applies to each Global Health Cover claim (except E-Opinion).'},
    {'id': 'F11', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the maximum total discount when discounts are combined?', 'sections': ['8'], 'evidence': [['shall not exceed 20']], 'facts': [['20%']], 'reference': 'Employee, online, loyalty and family discounts together cannot exceed 20%.'},
    {'id': 'F12', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What discount is given to NRI policyholders?', 'sections': ['8'], 'evidence': [['a discount of 40 subject to the']], 'facts': [['40%']], 'reference': 'Insured persons living abroad for the whole policy year get a 40% NRI discount.'},
    {'id': 'F13', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the moratorium period?', 'sections': ['12.6'], 'evidence': [['after completion of eight continuous years under the policy']], 'facts': [['eight continuous years', '8 continuous years', 'eight years', '8 years']], 'reference': 'After eight continuous years under the policy no look back applies, except for proven fraud and permanent exclusions.'},
    {'id': 'F14', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'Under which section of the Income Tax Act does the premium qualify for deduction?', 'sections': ['9'], 'evidence': [['deduction under section 80d of the income']], 'facts': [['80d']], 'reference': 'The premium qualifies for deduction under Section 80D of the Income Tax Act.'},
    {'id': 'F15', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How long is a cashless pre-authorization valid?', 'sections': ['11.5'], 'evidence': [['be valid for 15 days from the date']], 'facts': [['15 days']], 'reference': 'A pre-authorization is valid for 15 days from issue or until the policy expires, whichever is earlier.'},
    {'id': 'F16', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How far in advance must a planned hospitalization be notified?', 'sections': ['11.1'], 'evidence': [['at least 48 hours prior to admission']], 'facts': [['48 hours']], 'reference': 'A planned hospitalization must be notified at least 48 hours before admission.'},
    {'id': 'F17', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the maximum medical underwriting loading?', 'sections': ['13.4'], 'evidence': [['shall not exceed 100 for each condition']], 'facts': [['100%'], ['150%']], 'reference': 'Loading cannot exceed 100% per condition and 150% in total per insured person.'},
    {'id': 'F18', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How does the Cumulative Bonus work and which plan does it apply to?', 'sections': ['3.8'], 'evidence': [['applied increased by 10 of the base']], 'facts': [['10%'], ['optima suraksha'], ['100%']], 'reference': 'Cumulative Bonus applies only to the Optima Suraksha plan: 10% of the Base Sum Insured is added for each claim-free year, up to 100%, and reduced at the same rate after a claim.'},
    {'id': 'F19', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How early must a portability request be made?', 'sections': ['12.10'], 'evidence': [['any at least 45 days before but not']], 'facts': [['45 days'], ['60 days']], 'reference': 'Apply to port at least 45 days, but not earlier than 60 days, before the renewal date.'},
    {'id': 'F20', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'How much notice is needed for the policyholder to cancel the policy, and what is refunded?', 'sections': ['12.12'], 'evidence': [['person by giving 15 days written notice'], ['the insured a pro rata premium for the']], 'facts': [['15 days', '15days'], ['pro rata']], 'reference': "The policyholder can cancel with 15 days' written notice and gets a pro-rata refund for the unexpired period, unless a claim was admitted."},
    {'id': 'L01', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'Which expenses are covered under Hospitalization Expenses?', 'sections': ['3.1'], 'evidence': [['room rent boarding nursing expenses'], ['intensive care unit icu intensive'], ['surgeon anaesthetist medical practitioner'], ['medicines and drugs prescribed'], ['the cost of prosthetics and other devices']], 'facts': [['room rent'], ['intensive care', 'icu'], ['surgeon'], ['medicines'], ['prosthetics']], 'reference': 'Room rent and nursing, ICU/ICCU, surgeon and specialist fees, diagnostics, medicines, IV fluids and blood, operation theatre, and internally implanted prosthetics.'},
    {'id': 'L02', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'Which expenses are excluded from the Organ Donor Expenses cover?', 'sections': ['3.7'], 'evidence': [['organ donor s pre hospitalization expenses and post'], ['related to organ transportation or preservation'], ['consequent to the organ harvesting']], 'facts': [['pre hospitalization'], ['transportation or preservation', 'organ transportation'], ['organ harvesting']], 'reference': "The donor's pre- and post-hospitalization expenses, organ transportation or preservation, and other expenses consequent to the organ harvesting are excluded."},
    {'id': 'L03', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What are the specific exclusions for the Global Health Cover (Emergency Treatments Only)?', 'sections': ['4.9'], 'evidence': [['any planned treatments'], ['oncological cancer diseases'], ['any treatment of orthopedic diseases or conditions'], ['that is the sole reason or one of']], 'facts': [['planned treatments', 're:\\bplanned (?:treatment|procedure|surgery|surgeries)'], ['oncological', 'cancer'], ['orthopedic'], ['sole reason', 're:\\btravel(?:l?ing)?\\b[^.]{0,40}\\b(?:for|to (?:get|obtain|receive|seek)) (?:the )?(?:medical )?treatment', 'solely for', 'only for the purpose']], 'off_scope': ['4.10'], 'reference': 'Planned treatment, pre/post-hospitalization for a paid claim, non-life-threatening treatment that can wait, travel for treatment, orthopedic conditions except fractures and injuries, cancer, and countries under territorial restriction.'},
    {'id': 'L04', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'Which benefits does the Global Health Cover (Emergency & Planned Treatments) extend outside India?', 'sections': ['4.10'], 'evidence': [['extended for both planned and emergency medical expenses']], 'facts': [['ayush'], ['pre hospitalization'], ['post hospitalization'], ['organ donor'], ['air ambulance'], ['e opinion']], 'off_scope': ['4.9'], 'reference': 'Hospitalization, AYUSH, pre- and post-hospitalization, organ donor, emergency air ambulance, Protect Benefit, Plus Benefit and E-Opinion.'},
    {'id': 'L05', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What add-on covers are offered with my:Optima Secure?', 'sections': ['6', 'Annexure-A'], 'evidence': [['health critical illness add on with'], ['health hospital cash benefit add on with'], ['lndividual personal accident rider with sum'], ['unlimited restore add on']], 'facts': [['critical illness'], ['hospital cash'], ['personal accident'], ['unlimited restore']], 'reference': 'Critical Illness add-on, Hospital Cash Benefit add-on, Individual Personal Accident Rider and Unlimited Restore.'},
    {'id': 'L06', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What are the specific exclusions listed in Section 10.2?', 'sections': ['10.2'], 'evidence': [['war or any act'], ['injury or attempted suicide or suicide'], ['congenital external diseases defects or'], ['stem cell harvesting'], ['vaccination including inoculation and'], ['treatment taken on outpatient basis'], ['or fitting of hearing aids spectacles or contact']], 'facts': [['war'], ['suicide', 'self injury'], ['congenital external'], ['stem cell'], ['vaccination'], ['outpatient'], ['hearing aids', 'spectacles']], 'reference': 'War and nuclear risks, deductible amounts, self-injury and suicide, military operations, sleep-apnoea investigation, congenital external diseases, stem cell harvesting, spinal manipulation, circumcision, vaccination, non-medical items, outpatient treatment, hearing aids and spectacles, alopecia, external devices, unreasonable charges and permanent exclusions.'},
    {'id': 'L07', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'Which illnesses count as Major Medical Illnesses for the E-Opinion cover?', 'sections': ['4.8'], 'evidence': [['51 angioplasty', 'major medical illnesses 51 conditions'], ['cancer of specified severity'], ['kidney failure requiring regular dialysis'], ['stroke resulting in permanent']], 'facts': [['51'], ['cancer'], ['kidney failure'], ['stroke']], 'reference': '51 conditions, including cancer of specified severity, open chest CABG, kidney failure on dialysis, heart attack, stroke, major organ transplant and paralysis.'},
    {'id': 'L08', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What additional documents are needed for a claim outside India?', 'sections': ['11.7'], 'evidence': [['passport copy with entry'], ['flight tickets and boarding pass'], ['accommodation invoices if applicable'], ['written advice from the overseas']], 'facts': [['passport'], ['flight tickets', 'boarding pass'], ['accommodation invoices'], ['written advice']], 'reference': "Passport copy with entry and exit stamps, flight tickets and boarding pass, accommodation invoices, and the overseas doctor's written advice on the need for an accompanying person."},
    {'id': 'L09', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'Which cities are classified as Tier 1 for premium calculation?', 'sections': ['14'], 'evidence': [['delhi national capital region']], 'facts': [['delhi'], ['mumbai'], ['thane'], ['surat'], ['ahmedabad'], ['vadodara']], 'reference': 'Delhi and NCR, Mumbai and Mumbai Suburban, Thane and Navi Mumbai, Surat, Ahmedabad and Vadodara; the rest of India is Tier 2.'},
    {'id': 'L10', 'split': 'test', 'type': 'list', 'answerable': True, 'paraphrase': False, 'question': 'What can the company do if a pre-existing disease was not disclosed?', 'sections': ['13.1'], 'evidence': [['permanently exclude the disease condition'], ['of not exceeding 4 years for the said'], ['levy underwriting loading from the first']], 'facts': [['permanently exclude'], ['4 years'], ['loading']], 'reference': "With the policyholder's consent it can permanently exclude the condition, add a waiting period of up to 4 years, or apply underwriting loading."},
    {'id': 'P01', 'split': 'test', 'type': 'procedure', 'answerable': True, 'paraphrase': False, 'question': 'What is the procedure for cashless claims outside India?', 'sections': ['11.3'], 'evidence': [['we shall send release of information form to the'], ['issue the guarantee of payment specifying the sanctioned'], ['may be taken in a network provider and is subject to pre']], 'facts': [['release of information'], ['guarantee of payment'], ['network provider']], 'off_scope': ['11.2', '11.5'], 'reference': 'Intimate the claim, take treatment at a Network Provider subject to pre-authorization: the insured signs a Release of Information form, the Company retrieves the hospital documents and then issues a guarantee of payment or rejects the request with reasons.'},
    {'id': 'P02', 'split': 'test', 'type': 'procedure', 'answerable': True, 'paraphrase': False, 'question': 'How does cashless approval work for Home Health Care?', 'sections': ['11.4', '3.2'], 'evidence': [['will share the care plan and treatment cost'], ['issue the authorization letter specifying the sanctioned']], 'facts': [['care plan'], ['authorization letter']], 'reference': 'After a pre-authorization form is received, the home healthcare provider shares the care plan and cost estimate, and the Company issues an authorization letter with the sanctioned amount or rejects the request.'},
    {'id': 'P03', 'split': 'test', 'type': 'procedure', 'answerable': True, 'paraphrase': False, 'question': 'How and when must a claim be notified to the insurer?', 'sections': ['11.1'], 'evidence': [['within 24 hours from the date'], ['at least 48 hours prior to admission']], 'facts': [['24 hours'], ['48 hours']], 'reference': 'Notify within 24 hours of an emergency hospitalization (or before discharge, if earlier), and at least 48 hours before a planned admission or home health care.'},
    {'id': 'C01', 'split': 'dev', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'How does the maximum single-claim amount for the Optima Secure plan change from Year 1 to Year 3?', 'sections': ['2.1'], 'evidence': [['year 1 optima secure base sum insured', 'hospitalization claim year 1'], ['year 2 optima secure base sum insured', 'hospitalization claim year 2'], ['year 3 optima secure base sum insured', 'hospitalization claim year 3']], 'facts': [['20 lakh'], ['25 lakh'], ['30 lakh']], 'table_cells': [['secure', 'single', 1], ['secure', 'single', 2], ['secure', 'single', 3]], 'role': 'table: one plan across three years', 'reference': 'For Optima Secure the single-claim maximum is Rs. 20 lakh in Year 1, Rs. 25 lakh in Year 2 and Rs. 30 lakh in Year 3 (assuming no claims).'},
    {'id': 'C02', 'split': 'test', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'What is the difference between the Global Health Cover for emergency treatments only and the one for emergency and planned treatments?', 'sections': ['4.9', '4.10'], 'evidence': [['any planned treatments'], ['under this benefit pre hospitalization medical expenses and'], ['within the first 45 days of a trip']], 'facts': [['planned'], ['pre hospitalization'], ['45 days']], 'reference': 'The emergency-only cover excludes planned treatment and pre/post-hospitalization for overseas claims and requires treatment to start within 45 days of a trip; the emergency and planned cover also pays planned treatment and overseas pre- and post-hospitalization.'},
    {'id': 'C03', 'split': 'dev', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'How do the Cumulative Bonus and the Plus Benefit differ?', 'sections': ['3.8', '4.4'], 'evidence': [['applied increased by 10 of the base'], ['sum equal to 50 of the base'], ['will be applied irrespective of number of']], 'facts': [['10%'], ['50%'], ['claim free', 'no claim', 're:\\bwithout (a |any )?claims?\\b', 're:\\bno claims? (is |are |was |were )?(made|reported|filed)'], ['irrespective', 'regardless', 're:\\b(even|whether) (if|or not|though|when)\\b[^.]{0,50}\\bclaim', 're:\\bno matter\\b', 're:\\bnot (be )?(reduced|affected|impacted|dependent|linked|lost)\\b[^.]{0,40}\\bclaim', 're:\\b(does|do) not depend on\\b[^.]{0,20}\\bclaim', 're:\\bstill\\b[^.]{0,30}\\b(get|receive|added|eligible|entitled|apply|applies|be added)\\b']], 'role': 'comparison across two sections', 'reference': 'Cumulative Bonus (Optima Suraksha only) adds 10% per claim-free year and is reduced after a claim; Plus Benefit adds 50% per renewal irrespective of claims; both are capped at 100% of the Base Sum Insured.'},
    {'id': 'C04', 'split': 'test', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'How do the grace periods differ for monthly and yearly premium instalments?', 'sections': ['12.13'], 'evidence': [['monthly grace period applicable 15 days', 'monthly premium 15 days grace period'], ['yearly grace period applicable 30 days', 'yearly premium 30 days grace period']], 'facts': [['15 days'], ['30 days']], 'reference': 'Monthly instalments have a 15-day grace period; yearly, half-yearly and quarterly instalments have 30 days.'},
    {'id': 'K01', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'Can a 3-year-old child be insured without a parent on the policy?', 'sections': ['1'], 'evidence': [['either of the parent is getting insured']], 'facts': [['parent'], ['5 years']], 'polarity': 'no', 'reference': 'No. A child between 91 days and 5 years can be insured only if a parent is also insured; from 5 to 25 years a child can be insured individually.'},
    {'id': 'K02', 'split': 'dev', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'Will the air ambulance also fly me back home after treatment?', 'sections': ['4.1'], 'evidence': [['no return transportation to the insured']], 'facts': [['return transportation']], 'polarity': 'no', 'role': 'yes/no with a negative answer', 'reference': 'No. Return transportation to home or elsewhere by air ambulance is not covered.'},
    {'id': 'K03', 'split': 'dev', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'When can the Aggregate Deductible be waived or reduced?', 'sections': ['4.7.1'], 'evidence': [['be less than 50 years at the time'], ['after completion of 5 continuous policy years with']], 'facts': [['50 years', 're:\\b(?:aged?|ages|under|below|less than|younger than|up to|upto|until|till|before|over|above|at least|minimum of|maximum of|older than)(?: the)?(?: age)?(?: of)? 50\\b(?! (?:days|months|lakh))'], ['5 continuous'], ['61 years', 're:\\b(?:aged?|ages|under|below|less than|younger than|up to|upto|until|till|before|over|above|at least|minimum of|maximum of|older than)(?: the)?(?: age)?(?: of)? 61\\b(?! (?:days|months|lakh))']], 'role': 'conditions with three numbers', 'reference': "Once in the policy's lifetime at renewal, if the eldest insured was under 50 when buying, after 5 continuous policy years, and the eldest insured is under 61 when using the option."},
    {'id': 'K04', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'Is Daily Cash for Shared Room paid for time spent in the ICU?', 'sections': ['4.2'], 'evidence': [['person in an intensive care unit icu'], ['such hospitalization exceeds 48 consecutive hours']], 'facts': [['intensive care', 'icu']], 'polarity': 'no', 'reference': 'No. The daily cash is not paid for time spent in the ICU; it applies to shared rooms in Network Provider hospitals when hospitalization exceeds 48 hours.'},
    {'id': 'K05', 'split': 'dev', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'If I made claims last year, do I still get the extra 50% added when I renew?', 'sections': ['4.4'], 'evidence': [['sum equal to 50 of the base'], ['will be applied irrespective of number of']], 'facts': [['50%'], ['irrespective', 'regardless', 're:\\b(even|whether) (if|or not|though|when)\\b[^.]{0,50}\\bclaim', 're:\\bno matter\\b', 're:\\bnot (be )?(reduced|affected|impacted|dependent|linked|lost)\\b[^.]{0,40}\\bclaim', 're:\\b(does|do) not depend on\\b[^.]{0,20}\\bclaim', 're:\\bstill\\b[^.]{0,30}\\b(get|receive|added|eligible|entitled|apply|applies|be added)\\b']], 'polarity': 'yes', 'role': 'yes/no, everyday wording', 'reference': 'Yes. The Plus Benefit of 50% of the Base Sum Insured is added irrespective of the number of claims, up to 100%.'},
    {'id': 'K06', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'Does unused Secure Benefit carry forward to the next year?', 'sections': ['4.5'], 'evidence': [['or in part will not be carried forward to the subsequent policy']], 'facts': [['not be carried forward', 'not carried forward']], 'polarity': 'no', 'reference': 'No. Any unused Secure Benefit is not carried forward; it applies once per policy year after the Base Sum Insured is exhausted.'},
    {'id': 'K07', 'split': 'dev', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'Under what conditions will the policy pay for bariatric surgery?', 'sections': ['10.1'], 'evidence': [['has to be 18 years of age or'], ['or equal to 40 or b'], ['or equal to 35 in conjunction with']], 'facts': [['18 years', 're:\\b(?:aged?|ages|under|below|less than|younger than|up to|upto|until|till|before|over|above|at least|minimum of|maximum of|older than)(?: the)?(?: age)?(?: of)? 18\\b(?! (?:days|months|lakh))', 're:\\b18 (?:years )?(?:or|and) (?:older|above|more)\\b'], ['re:\\b40\\b'], ['re:\\b35\\b'], ['morbidit']], 'role': 'conditions, everyday wording', 'reference': "Obesity surgery (Excl06) is covered only on a doctor's advice, supported by clinical protocols, for members aged 18 or older with BMI of 40 or more, or 35 or more with a severe co-morbidity after less invasive methods failed."},
    {'id': 'K08', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'Does the 30-day initial waiting period still apply if I have been continuously insured for more than a year?', 'sections': ['10.1'], 'evidence': [['for more than twelve months']], 'facts': [['twelve months', '12 months']], 'polarity': 'no', 'reference': 'No. The 30-day exclusion does not apply if the insured has continuous coverage for more than twelve months.'},
    {'id': 'K09', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': True, 'question': 'Can the insurer refuse to renew my policy because I made claims?', 'sections': ['12.9'], 'evidence': [['renewal shall not be denied on the ground']], 'facts': [['not be denied']], 'polarity': 'no', 'reference': 'No. Renewal shall not be denied because of claims made in earlier years; it can be refused only for fraud or misrepresentation.'},
    {'id': 'K10', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'What happens to the Plus Benefit if the Sum Insured is reduced at renewal?', 'sections': ['4.4'], 'evidence': [['shall be reduced in the same proportion to the sum insured in']], 'facts': [['same proportion']], 'reference': 'The Plus Benefit is reduced in the same proportion as the Sum Insured.'},
    {'id': 'K11', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'On a family floater policy, how is the Cumulative Bonus added and when is it reduced?', 'sections': ['3.8'], 'evidence': [['policy is on floater basis the cb shall']], 'facts': [['floater basis', 're:\\b(?:whole|entire) family\\b', 'family as a whole', 'floater sum insured'], ['any family member', 'any of the insured persons']], 'reference': 'On a floater policy the Cumulative Bonus is added for the whole family, provided no claim is reported from any family member, and it is reduced after a claim from any insured person.'},
    {'id': 'K12', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'What happens to an awarded Cumulative Bonus if a claim from the expiring year is notified after the renewal premium is accepted?', 'sections': ['3.8'], 'evidence': [['cb shall be withdrawn']], 'facts': [['withdrawn']], 'reference': 'Any awarded Cumulative Bonus is withdrawn.'},
    {'id': 'D01', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is the Protect Benefit?', 'sections': ['4.3'], 'evidence': [['person for the non medical expenses listed under annexure']], 'facts': [['non medical expenses'], ['annexure b']], 'reference': 'Protect Benefit pays the non-medical expenses listed in Annexure B for claims admissible under Section 3.'},
    {'id': 'D02', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is the Automatic Restore Benefit?', 'sections': ['4.6'], 'evidence': [['utilization of the base sum insured due to any'], ['restore benefit shall be applied only once during each policy year and any']], 'facts': [['base sum insured'], ['once'], ['not be carried forward', 'not carried forward', 're:\\b(?:not|never|cannot|can not|won t|will not) (?:be )?carr(?:y|ied) (?:it )?(?:forward|over)\\b', 're:\\bdoes not carry (?:forward|over)\\b']], 'reference': 'After a claim uses the Base Sum Insured in whole or part, it is restored up to the Base Sum Insured for later claims, once per policy year, and unused amounts are not carried forward.'},
    {'id': 'D03', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is domiciliary hospitalization and when is it covered?', 'sections': ['3.3'], 'evidence': [['he she could not be removed admitted to a'], ['on account of non availability of room in a hospital']], 'facts': [['not be removed', 're:\\b(?:not|cannot|can not|could not|unable to) be (?:moved|removed|shifted|taken|transferred)\\b', 're:\\binability to be (?:moved|removed|shifted|admitted|taken)\\b'], ['non availability of room', 're:\\bno (?:hospital )?(?:room|bed)s? (?:was |is |are |were )?available', 're:\\b(?:non|un)availability of (?:a )?(?:hospital )?(?:room|bed)', 're:\\b(?:room|bed)s? (?:is|are|was|were) not available', 're:\\b(?:non ?|un)availability of (?:a )?(?:hospital )?(?:room|bed)s?\\b']], 'reference': 'Treatment at home, prescribed by a doctor, when the patient could not be moved to a hospital or no hospital room was available.'},
    {'id': 'D04', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What does the AYUSH Treatment cover include?', 'sections': ['3.4'], 'evidence': [['inpatient care under ayurveda yoga and naturopathy']], 'facts': [['ayurveda'], ['homeopathy'], ['sub limit']], 'reference': 'In-patient care under Ayurveda, Yoga and Naturopathy, Unani, Siddha and Homeopathy in an AYUSH hospital, up to the sub-limit in the Policy Schedule.'},
    {'id': 'D05', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is the E-Opinion for Critical Illness cover?', 'sections': ['4.8'], 'evidence': [['insured person only once in a policy'], ['below through the network provider specified in the'], ['impact on the sum insured']], 'facts': [['once'], ['network provider'], ['sum insured']], 'reference': 'An electronic second opinion on a listed major illness through the Network Provider, once per policy year per insured person, with no impact on the Sum Insured.'},
    {'id': 'D06', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is the Preventive Health Check-up benefit?', 'sections': ['5'], 'evidence': [['on each continuous renewal of the policy'], ['this cover does not carry forward if it is']], 'facts': [['renewal'], ['not carry forward']], 'reference': 'On each continuous renewal the cost of a health check-up is reimbursed up to the specified limit; it does not carry forward if unclaimed.'},
    {'id': 'D07', 'split': 'test', 'type': 'definition', 'answerable': True, 'paraphrase': False, 'question': 'What is Home Health Care cover?', 'sections': ['3.2'], 'evidence': [['the treatment is pre authorized by the company'], ['this cover is not available on reimbursement basis']], 'facts': [['pre authorized'], ['not available on reimbursement', 're:\\bnot (?:available|payable|paid) on (?:a )?reimbursement\\b']], 'reference': 'Treatment at home that would otherwise need in-patient care, if prescribed and pre-authorized, with daily signed records; it is not available on reimbursement basis.'},
    {'id': 'M01', 'split': 'dev', 'type': 'multi_section', 'answerable': True, 'paraphrase': True, 'question': 'In what order is the Sum Insured used up when I make claims?', 'sections': ['13.2'], 'evidence': [['the sequence of utilization of the sum insured']], 'facts': [['base sum insured'], ['plus benefit', 'cumulative bonus'], ['secure benefit'], ['automatic restore']], 'ordered': True, 'must_not': ['base sum insured (is|was|will be) not (utilized|utilised|used)'], 'role': 'order of use; right words, wrong meaning', 'reference': 'Base Sum Insured first, then Cumulative Bonus or Plus Benefit, then Secure Benefit, then Automatic Restore.'},
    {'id': 'M02', 'split': 'test', 'type': 'multi_section', 'answerable': True, 'paraphrase': False, 'question': 'Which covers are not subject to the Aggregate Deductible?', 'sections': ['4.7', '4.9', '4.10'], 'evidence': [['sections 4 8 e opinion for critical illness']], 'facts': [['e opinion'], ['preventive health'], ['global health cover'], ['overseas travel secure']], 'reference': 'E-Opinion for Critical Illness, Preventive Health Check-up, both Global Health Covers and Overseas Travel Secure.'},
    {'id': 'M03', 'split': 'test', 'type': 'multi_section', 'answerable': True, 'paraphrase': True, 'question': 'Does this policy cover treatment taken outside India?', 'sections': ['13.3', '4.9', '4.10'], 'evidence': [['section 4 9 global health cover emergency treatments only section 4 10'], ['the territory of india except under section']], 'facts': [['global health cover'], ['india']], 'reference': 'The policy covers India only, except through the optional Global Health Covers (Sections 4.9 and 4.10), Overseas Travel Secure and E-Opinion.'},
    {'id': 'M04', 'split': 'test', 'type': 'multi_section', 'answerable': True, 'paraphrase': False, 'question': 'What limits and conditions apply to claims under the Global Health Cover?', 'sections': ['4.9', '4.10'], 'evidence': [['shall not exceed the base sum insured and plus benefit if available'], ['10 000 will apply separately for each and every claim except section'], ['claims shall normally be payable on reimbursement basis only'], ['be based on the rate of exchange as on the date of payment']], 'facts': [['base sum insured and plus benefit', 're:\\bbase sum insured (?:and|plus|\\+) (?:the )?plus benefit'], ['10,000'], ['reimbursement', 're:\\breimburs'], ['exchange']], 'reference': 'Liability is capped at the Base Sum Insured plus Plus Benefit, a Rs. 10,000 deductible applies per claim, claims are normally reimbursed (cashless case by case) at the RBI exchange rate, in registered hospitals.'},
    {'id': 'X01', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'How long do I have to wait before kidney stone treatment is covered?', 'sections': ['10.1'], 'evidence': [['the expiry of 24 months of continuous coverage']], 'facts': [['24 months']], 'reference': 'Kidney stones are a listed condition under Excl02, so treatment is covered after 24 months of continuous coverage, unless caused by an accident.', 'must_not': ['regardless of (?:any |an |the )?accidents?', '\\b(?:applies|apply|applicable) (?:even )?(?:to|for|in case of) (?:claims (?:arising )?(?:due to|from) )?(?:an )?accidents?']},
    {'id': 'X02', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Is cataract surgery covered from day one?', 'sections': ['10.1'], 'evidence': [['the expiry of 24 months of continuous coverage']], 'facts': [['24 months']], 'polarity': 'no', 'reference': 'No. Cataract is a listed condition with a 24-month waiting period (Excl02).'},
    {'id': 'X03', 'split': 'dev', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': "Will the policy pay for my wife's delivery?", 'sections': ['10.1'], 'evidence': [['excl18'], ['expenses traceable to childbirth including complicated deliveries']], 'facts': [['excl18', 'maternity'], ['childbirth']], 'polarity': 'no', 'role': 'exclusion in everyday wording', 'reference': 'No. Childbirth expenses, including caesarean sections, are excluded under Excl18 (Maternity), except ectopic pregnancy.'},
    {'id': 'X04', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Are IVF treatments reimbursed?', 'sections': ['10.1'], 'evidence': [['excl17']], 'facts': [['excl17', 'infertility', 'sterility']], 'polarity': 'no', 'reference': 'No. Assisted reproduction such as IVF is excluded under Excl17 (Sterility and Infertility).'},
    {'id': 'X05', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'I got hurt while scuba diving as a professional diver. Will I be covered?', 'sections': ['10.1'], 'evidence': [['excl09']], 'facts': [['excl09', 'hazardous', 'adventure']], 'polarity': 'no', 'reference': 'No. Treatment needed because of professional participation in hazardous or adventure sports, including scuba diving, is excluded (Excl09).'},
    {'id': 'X06', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Can I claim the cost of new spectacles?', 'sections': ['10.2'], 'evidence': [['of hearing aids spectacles or contact lenses']], 'facts': [['spectacles']], 'polarity': 'no', 'reference': 'No. Provision or fitting of hearing aids, spectacles or contact lenses is excluded.'},
    {'id': 'X07', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'For how many days before admission is pre hospitalisation covered?', 'sections': ['3.5'], 'evidence': [['for up to 60 days immediately prior to']], 'facts': [['60 days']], 'off_scope': ['4.9', '4.10'], 'reference': 'Up to 60 days before an admissible hospitalization.'},
    {'id': 'X08', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'How many days of post hospitalisation expenses can I claim after discharge?', 'sections': ['3.6'], 'evidence': [['for up to 180 days from the date']], 'facts': [['180 days']], 'off_scope': ['4.9', '4.10'], 'reference': 'Up to 180 days from the date of discharge.'},
    {'id': 'X09', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Do I pay less if I buy the policy on the website?', 'sections': ['8'], 'evidence': [['is eligible for 5 discount on premium']], 'facts': [['5%']], 'polarity': 'yes', 'reference': 'Yes. Buying online gives a 5% discount, which continues on renewals without an intermediary.'},
    {'id': 'X10', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'My son is turning 25. What happens to his cover?', 'sections': ['1'], 'evidence': [['and will be migrated to a new']], 'facts': [['migrated', 'new policy'], ['continuity']], 'reference': 'At 25 he becomes ineligible on the next renewal and is migrated to a new policy with continuity benefits.'},
    {'id': 'X11', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Does the policy pay for an ambulance to take me to the hospital?', 'sections': ['3.1.1'], 'evidence': [['expenses incurred on road ambulance if the insured']], 'facts': [['road ambulance', 'ambulance'], ['nearest hospital']], 'polarity': 'yes', 'reference': 'Yes. Road ambulance costs are covered for transfer to the nearest hospital for emergency care, between hospitals, or from hospital to home in the same city.'},
    {'id': 'X12', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Is dental treatment covered?', 'sections': ['3.1.1'], 'evidence': [['in patient care dental treatment necessitated due']], 'facts': [['dental'], ['disease or injury']], 'polarity': 'yes', 'reference': 'In-patient dental treatment is covered when necessitated by disease or injury.'},
    {'id': 'X13', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'How soon will I get paid after submitting my claim papers?', 'sections': ['12.3'], 'evidence': [['may be within 30 days from the date']], 'facts': [['30 days']], 'reference': 'Within 30 days of the last necessary document, or 45 days if investigated, with interest at 2% above the bank rate for delays.'},
    {'id': 'X14', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'If I move my policy to another insurer, do I keep the waiting period I have already served?', 'sections': ['12.10'], 'evidence': [['insured person will get the accrued continuity benefits in waiting periods as per']], 'facts': [['continuity']], 'polarity': 'yes', 'reference': 'Yes. With continuous coverage, the accrued continuity benefits in waiting periods carry over as per IRDAI portability guidelines.'},
    {'id': 'X15', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Can I pay my premium every month?', 'sections': ['12.13'], 'evidence': [['yearly quarterly or monthly as mentioned in'], ['monthly grace period applicable 15 days', 'monthly premium 15 days grace period']], 'facts': [['monthly'], ['15 days']], 'polarity': 'yes', 'reference': 'Yes. Monthly instalments are allowed, with a 15-day grace period during which there is no cover.'},
    {'id': 'N01', 'split': 'dev', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'Does this policy cover damage to my car in an accident?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['vehicle damage', 'car damage', 'own damage'], 'role': 'not answerable, unrelated', 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N02', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'How much does the policy pay for lost baggage during travel?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['baggage', 'luggage'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N03', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'What is the interest rate on a personal loan from HDFC?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['loan'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N04', 'split': 'dev', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'What does the Optima Restore plan cover?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['optima restore'], 'role': 'not answerable, similar product name', 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N05', 'split': 'dev', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the annual premium for a 40-year-old living in a Tier 2 city?', 'sections': ['Annexure-C'], 'evidence': [['age 40 10 300 13 300 16 000'], ['age 40 10 622 13 579 16 294']], 'facts': [['re:(?<![\\d,.])(?:10,?300|13,?300|16,?000|16,?650|17,?250|20,?150|23,?000|28,?000|10,?622|13,?579|16,?294|16,?908|17,?473|20,?356|23,?180|28,?210)(?![\\d,]*\\d)']], 'must_not': ['(?<![\\d,.])(?:12,?500|15,?500|16,?400|17,?200|17,?950|21,?000|25,?000|32,?000|12,?891|15,?825|16,?701|17,?467|18,?182|21,?215|25,?196|32,?240|10,?200|13,?150|15,?800|16,?450|17,?050|19,?950|22,?800|27,?750|10,?400|13,?450|16,?200|16,?850|17,?450|20,?350|23,?200|28,?250|10,?519|13,?425|16,?090|16,?705|17,?271|20,?154|22,?978|27,?958|10,?725|13,?732|16,?497|17,?112|17,?676|20,?558|23,?382|28,?462)(?![\\d,]*\\d)'], 'role': 'premium table lookup (separate premium index)', 'reference': 'It depends on the plan and Sum Insured (Annexure C, Tier 2, age 40, excluding GST): Optima Secure from Rs. 10,300 (Rs. 5 lakh) to Rs. 28,000 (Rs. 2 crore); Optima Super Secure from Rs. 10,622 to Rs. 28,210.'},
    {'id': 'N06', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'Does the policy cover home contents against fire?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['fire', 'home contents', 'burglary'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'FP01', 'split': 'dev', 'type': 'false_premise', 'answerable': False, 'paraphrase': False, 'question': 'What is the co-payment percentage for senior citizens?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['co-payment for senior', 'senior citizen co-pay'], 'role': 'false premise, not answerable', 'reference': 'Not specified in the provided policy text: no senior-citizen co-payment is stated.'},
    {'id': 'FP02', 'split': 'test', 'type': 'false_premise', 'answerable': False, 'paraphrase': False, 'question': 'What is the room rent limit of 1% of the Sum Insured per day?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['1% of the sum insured', 'room rent limit'], 'reference': 'Not specified in the provided policy text: no room rent cap of 1% of the Sum Insured is stated.'},
    {'id': 'FP03', 'split': 'dev', 'type': 'false_premise', 'answerable': True, 'paraphrase': True, 'question': 'How much is the maternity benefit after the 9-month waiting period?', 'sections': ['10.1'], 'evidence': [['excl18']], 'facts': [['excl18', 'excluded', 'not covered']], 'role': 'false premise, answerable (no such benefit)', 'reference': 'There is no maternity benefit: childbirth expenses are excluded under Excl18, and no 9-month waiting period is stated.'},
    {'id': 'T01', 'split': 'dev', 'type': 'comparison', 'answerable': True, 'paraphrase': False, 'question': 'What is the maximum amount payable for all hospitalization claims in a policy year under Optima Super Secure in Year 2?', 'sections': ['2.1'], 'evidence': [['year 2 optima super secure base sum insured', 'hospitalization claim year 2']], 'facts': [['45 lakh']], 'table_cells': [['super', 'all', 2]], 'role': 'table: one cell', 'reference': 'Rs. 45 lakh (Year 2 illustration: Base Sum Insured Rs. 10 lakh, no claim in Year 1).'},
    {'id': 'TY01', 'split': 'dev', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'wat is the free look peroid for a new polcy?', 'sections': ['12.8'], 'evidence': [['look period of fifteen days from date of']], 'facts': [['fifteen days', '15 days']], 'role': 'typos in the question', 'reference': 'Fifteen days from receipt of the policy document, for new individual policies.'},
    {'id': 'PA01', 'split': 'dev', 'type': 'partial', 'answerable': True, 'paraphrase': True, 'question': 'What discount do I get for buying online, and is there a special discount for women?', 'sections': ['8'], 'evidence': [['is eligible for 5 discount on premium']], 'facts': [['5%']], 'expect_admit': True, 'absent_terms': ['women', 'woman', 'female'], 'role': 'half answerable: must say what is not shown', 'reference': '5% for buying online (kept on renewal without an intermediary); the policy does not mention a discount for women.'},
    {'id': 'PR01', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'What is the annual premium for Optima Secure with a Rs. 10 lakh Sum Insured for a 30-year-old living in Mumbai?', 'sections': ['Annexure-C', '14'], 'evidence': [['age 30 11 000 13 500 14 400']], 'facts': [['re:(?<![\\d,.])13,?500(?![\\d,]*\\d)']], 'must_not': ['(?<![\\d,.])(?:11,?800|13,?300|13,?700|11,?000|14,?400|13,?783)(?![\\d,]*\\d)'], 'reference': 'Rs. 13,500 a year (Mumbai is Tier 1; age 30; Rs. 10 lakh; Annexure C, excluding GST).'},
    {'id': 'PR02', 'split': 'test', 'type': 'comparison', 'answerable': True, 'paraphrase': True, 'question': 'How much more does Optima Super Secure cost than Optima Secure for a 45-year-old living in Pune with Rs. 5 lakh cover?', 'sections': ['Annexure-C', '14'], 'evidence': [['age 45 10 800 14 050 17 000'], ['age 45 11 138 14 344 17 312'], ['rest of india no co payment']], 'facts': [['re:(?<![\\d,.])10,?800(?![\\d,]*\\d)'], ['re:(?<![\\d,.])11,?138(?![\\d,]*\\d)'], ['re:(?<![\\d,.])338(?![\\d,]*\\d)']], 'must_not': ['(?<![\\d,.])(?:13,?250|13,?665|10,?700|11,?800|11,?035|12,?169)(?![\\d,]*\\d)'], 'derived_figures': [['inr', 338]], 'reference': 'Pune is not a Tier 1 city, so Tier 2 (Rest of India) applies: Rs. 11,138 for Optima Super Secure vs Rs. 10,800 for Optima Secure, Rs. 338 more a year (excluding GST).'},
    {'id': 'PR03', 'split': 'test', 'type': 'fact', 'answerable': True, 'paraphrase': False, 'question': 'Do the premium rates in the policy include GST?', 'sections': ['14', 'Annexure-C'], 'evidence': [['premium table exclusive of goods and services tax', 'annual gross premium excl gst']], 'facts': [['excl. gst', 'exclusive of goods and services tax', 'exclusive of gst', 'excluding gst', 'excludes gst', 're:\\b(do|does) not include (the )?(gst|goods and services tax)', 're:\\bwithout (the )?gst\\b', 're:\\b(gst|goods and services tax) (is |will be )?(charged |added |payable )?(extra|separately|additionally|excluded)']], 'polarity': 'no', 'reference': 'No. The premium table (Annexure C) is exclusive of Goods and Services Tax (GST).'},
    {'id': 'AB01', 'split': 'test', 'type': 'conditional', 'answerable': True, 'paraphrase': False, 'question': 'Are gloves and food charges paid in a hospitalization claim?', 'sections': ['Annexure-B', '4.3', '0'], 'evidence': [['ecg electrodes 56 gloves', 'non medical expenses like gloves food charges'], ['non medical expenses listed under annexure b']], 'facts': [['protect benefit'], ['annexure b', 'not payable', 'not covered', 'not available', 'otherwise not payable', 'non medical', 'not reimbursed', 'not reimbursable', 'excluded']], 'reference': "Not under the base cover: gloves and food charges (other than the patient's diet) are on the Annexure B list of non-medical items not covered. The optional Protect Benefit pays the Annexure B items for claims admissible under Section 3."},
    {'id': 'AB02', 'split': 'test', 'type': 'paraphrase', 'answerable': True, 'paraphrase': True, 'question': 'Will I be reimbursed for attendant charges and the TV in my hospital room?', 'sections': ['Annexure-B', '4.3'], 'evidence': [['22 television charges 23 surcharges 24 attendant charges'], ['non medical expenses listed under annexure b']], 'facts': [['annexure b', 'not payable', 'not covered', 'not available', 'otherwise not payable', 'non medical', 'not reimbursed', 'not reimbursable', 'excluded'], ['protect benefit']], 'reference': 'Not under the base cover: attendant charges and television charges are Annexure B non-medical items; they are paid only if the optional Protect Benefit is in force, for claims admissible under Section 3.'},
    {'id': 'FP04', 'split': 'test', 'type': 'false_premise', 'answerable': False, 'paraphrase': False, 'question': 'What is the 10% co-payment that applies to every claim?', 'sections': [], 'evidence': [], 'facts': [], 'must_not': ['\\b10 ?% co payment (applies|is applicable|is charged|will apply|is deducted)'], 'absent_terms': ['10% co-payment', 'co-payment of 10'], 'reference': 'Not specified: the policy states no co-payment on every claim; it only says no co-payment applies when a Tier 2 insured person is treated in a Tier 1 city.'},
    {'id': 'FP05', 'split': 'test', 'type': 'false_premise', 'answerable': False, 'paraphrase': False, 'question': 'What is the sub-limit for cataract surgery per eye?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['per eye', 'cataract sub-limit', 'sub-limit for cataract'], 'reference': 'Not specified: no cataract sub-limit is stated (cataract is a listed condition with a 24-month waiting period, Excl02).'},
    {'id': 'N07', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': "What is HDFC ERGO's claim settlement ratio?", 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['settlement ratio'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N08', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': True, 'question': 'Which hospitals in Pune are in the cashless network?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['pune'], 'reference': "Not specified: the policy does not list network hospitals (it says the list of Network Providers is on the company's website)."},
    {'id': 'N09', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'Does the policy cover trip cancellation or flight delays?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['trip cancellation', 'flight delay', 'cancellation of trip'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N10', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'What GST rate is charged on the premium?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['18%', 'gst rate', 'rate of gst'], 'reference': 'Not specified: the premium table is shown excluding GST and no GST rate is stated.'},
    {'id': 'N11', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': True, 'question': 'How many free doctor teleconsultations do I get each year?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['teleconsult', 'tele consultation', 'tele-consultation', 'video consultation'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N12', 'split': 'dev', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'What is the annual premium for Optima Suraksha for a 30-year-old?', 'sections': [], 'evidence': [], 'facts': [], 'must_not': ['(premium|cost|price)s? (for|of) (the )?(optima )?suraksha[^.\\n]{0,60}\\b(is|are|would be|will be)\\b[^.\\n]{0,15}\\d'], 'absent_terms': ['optima suraksha, tier'], 'role': 'not answerable: premium of a plan the tables do not cover', 'reference': 'Not specified: the premium table (Annexure C) covers only Optima Secure and Optima Super Secure.'},
    {'id': 'N13', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': False, 'question': 'Can this policy be converted into a life insurance policy?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['life insurance'], 'reference': 'Not specified in the provided policy text.'},
    {'id': 'N14', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': True, 'question': 'Do I get my premium back as cashback at maturity if I make no claims?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['maturity', 'cashback', 'cash back', 'return of premium'], 'reference': 'Not specified: the policy describes no maturity or premium-return benefit.'},
    {'id': 'N15', 'split': 'test', 'type': 'out_of_scope', 'answerable': False, 'paraphrase': True, 'question': 'Does the plan include a free gym membership?', 'sections': [], 'evidence': [], 'facts': [], 'absent_terms': ['gym', 'fitness'], 'reference': 'Not specified in the provided policy text.'},

]
TESTSET_VERSION = hashlib.sha1(json.dumps(TESTSET, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:10]
BY_ID = {t["id"]: t for t in TESTSET}
ITEMS = [t for t in TESTSET if t["split"] in SETTINGS["splits"]]

# ---------------------------------------------------------------- matching (answer key <-> text)
def norm_text(s: str) -> str:
    """Lower-case; drop currency marks; dashes and slashes to spaces; -isation/-ised to -ization/-ized;
    "pre- and post-hospitalization" spelled out."""
    s = (s or "").lower().replace("₹", " ")
    s = re.sub(r"\bpre\s*[-‐‑–]?\s*(?:and|or|&|/)\s*post\s*[-‐‑–]?\s*(hospitali[sz](?:ation|ed))",
               r"pre-\1 and post-\1", s)                    # "pre- and post-hospitalization": both words
    s = re.sub(r"\brs\.?\s*", " ", s)
    s = re.sub(r"[-‐‑‒–—/]", " ", s)
    s = re.sub(r"isation", "ization", s)
    s = re.sub(r"ised\b", "ized", s)
    return re.sub(r"\s+", " ", s).strip()

def _alt_re(alt: str) -> str:
    """A phrase matches at a word start ("5 years" is not found inside "65 years"); a phrase ending in a
    digit must also end there ("10,000" is not found inside "10,000,000"); "N days/months/years" also
    matches the singular ("a 30-day period")."""
    if alt.startswith("re:"):
        return alt[3:]
    a = norm_text(alt)
    if re.search(r"\d (day|week|month|year|hour)s$", a):     # "30 days" also matches "30-day"
        return r"(?<![0-9a-z.,])" + re.escape(a[:-1]) + r"s?(?![a-z])"
    return r"(?<![0-9a-z.,])" + re.escape(a) + (r"(?![0-9]|[.,][0-9])" if a[-1:].isdigit() else "")

def fact_pos(alternatives, text_norm: str):
    """Position of the first alternative found, or None."""
    hits = [m.start() for m in (re.search(_alt_re(a), text_norm) for a in alternatives) if m]
    return min(hits) if hits else None

def fact_found(alternatives, text_norm: str) -> bool:
    return fact_pos(alternatives, text_norm) is not None

def fact_label(f) -> str:
    return re.sub(r"\(\?[<!=][^)]*\)|\\[bd]|re:|[?(){}\[\]\\^$*+|]", "", f[0])[:40]

def _words(s: str) -> str:
    return " " + " ".join(re.findall(r"[a-z0-9]+", norm_text(s))) + " "

def anchor_in(alternatives, words_text: str) -> bool:
    """An evidence passage is in a text (as returned by _words) if one anchor's words occur in order; anchors of 6+
    words also match on 60% of their words in a row (a chunk boundary may cut them)."""
    for a in alternatives:
        w = _words(a).split()
        if " " + " ".join(w) + " " in words_text:
            return True
        n = max(4, math.ceil(0.6 * len(w)))
        if len(w) >= 6 and any(" " + " ".join(w[i:i + n]) + " " in words_text for i in range(len(w) - n + 1)):
            return True
    return False

for c in final_chunks:
    c["words"] = _words(c["text"])
_CORPUS_NORM = norm_text(" ".join(c["label"] + " " + c["text"] for c in final_chunks))
_CORPUS_WORDS = " ".join(c["words"] for c in final_chunks)

def check_testset(items):
    """Facts and evidence against the loaded chunk file; absent terms really absent. Missing evidence is
    expected for the old text file (no premium tables, Annexure B list missing)."""
    rows = []
    for t in items:
        if t["answerable"]:
            miss_f = [fact_label(f) for f in t["facts"] if not fact_found(f, _CORPUS_NORM)
                      and not any(fact_found(f, str(v)) for _, v in t.get("derived_figures", []))]
            miss_e = sum(not anchor_in(e, _CORPUS_WORDS) for e in t["evidence"])
            if miss_f or miss_e:
                rows.append({"id": t["id"], "problem": "evidence not in this chunk file",
                             "detail": f"{miss_e}/{len(t['evidence'])} passages; facts: {', '.join(miss_f) or '-'}"})
        for term in t.get("absent_terms", []):
            if not t["answerable"] and re.search(r"\b" + re.escape(norm_text(term)) + r"\b", _CORPUS_NORM):
                rows.append({"id": t["id"], "problem": "out-of-scope term occurs in the chunk file", "detail": term})
    return pd.DataFrame(rows, columns=["id", "problem", "detail"])

TESTSET_PROBLEMS = check_testset(ITEMS)
EVIDENCE_MISSING = set(TESTSET_PROBLEMS.query("problem == 'evidence not in this chunk file'")["id"])
_pdf_ids = sum(all(s in SECTIONS for s in t["sections"]) for t in ITEMS if t["answerable"])
_n_ans = sum(t["answerable"] for t in ITEMS)
print(f"Test set {TESTSET_VERSION}: {len(TESTSET)} questions; running {len(ITEMS)} "
      f"({', '.join(f'{s} {sum(t['split'] == s for t in ITEMS)}' for s in SETTINGS['splits'])}): "
      f"{_n_ans} answerable, {len(ITEMS) - _n_ans} not answerable, {sum(t['paraphrase'] for t in ITEMS)} in everyday wording")
print(f"Section numbers of this chunk file match the PDF for {_pdf_ids}/{_n_ans} answerable questions"
      + ("" if _pdf_ids == _n_ans else " (citation and scope checks use PDF numbering)"))
if len(TESTSET_PROBLEMS):
    print(TESTSET_PROBLEMS.to_string(index=False))
else:
    print("All facts and evidence passages found; all out-of-scope terms absent.")


## RAG pipeline

In [ ]:
# ================================================================
# 5) Models: embedder, reranker, answer model (first run downloads ~20 GB, about 5-8 min)
# ================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
NEED_LLM = SETTINGS["mode"] == "full"

def revision(model, name: str = None):
    """The model's commit on the Hugging Face Hub (recorded so a result can be traced to exact weights)."""
    for cfg in (getattr(model, "config", None), getattr(getattr(model, "auto_model", None), "config", None)):
        h = getattr(cfg, "_commit_hash", None)
        if h:
            return h
    if name and not Path(name).exists():
        try:
            from huggingface_hub import HfApi
            return HfApi().model_info(name).sha
        except Exception:
            return None
    return None

# The answer model's tokenizer also counts the context budget, so it loads in every mode.
tok_llm = AutoTokenizer.from_pretrained(SETTINGS["llm_model"])
if tok_llm.pad_token is None:
    tok_llm.pad_token = tok_llm.eos_token

def tok_len(s: str) -> int:
    return len(tok_llm(s, add_special_tokens=False)["input_ids"])

embed_model = SentenceTransformer(SETTINGS["embed_model"], device=DEVICE)
tok_rerank = AutoTokenizer.from_pretrained(SETTINGS["reranker_model"])
reranker = AutoModelForSequenceClassification.from_pretrained(
    SETTINGS["reranker_model"], **dtype_kw(torch.float16 if DEVICE == "cuda" else torch.float32)).to(DEVICE).eval()

model_llm, LLM_PRECISION, _offloaded = None, None, []
if NEED_LLM:
    kw = {"device_map": "auto"} if DEVICE == "cuda" else {}
    if SETTINGS["llm_4bit"] and DEVICE == "cuda":
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True,
                                                       bnb_4bit_compute_dtype=torch.float16)   # T4: no bfloat16
        LLM_PRECISION = "4-bit NF4"
    else:
        LLM_PRECISION = "fp16" if DEVICE == "cuda" else "fp32 (CPU)"
    model_llm = AutoModelForCausalLM.from_pretrained(
        SETTINGS["llm_model"], **dtype_kw(torch.float16 if DEVICE == "cuda" else torch.float32), **kw).eval()
    _offloaded = [k for k, v in (getattr(model_llm, "hf_device_map", None) or {}).items() if v in ("cpu", "disk")]
    if _offloaded:
        log(f"{len(_offloaded)} answer-model modules are on the CPU/disk (GPU full): answers will be very slow")

def chat_text(user_msg: str, thinking=None) -> str:
    """The prompt as the model sees it. enable_thinking switches Qwen3's thinking mode; other templates ignore it."""
    thinking = SETTINGS["enable_thinking"] if thinking is None else thinking
    return tok_llm.apply_chat_template([{"role": "user", "content": user_msg}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=thinking)

def llm_inputs(user_msg: str, thinking=None):
    text = chat_text(user_msg, thinking)
    enc = tok_llm(text, return_tensors="pt", add_special_tokens=False)
    bos = tok_llm.bos_token_id
    if bos is not None and tok_llm.bos_token and not text.startswith(tok_llm.bos_token):
        enc = tok_llm(text, return_tensors="pt")          # template without BOS: add exactly one
    return enc.to(model_llm.device)

# Decoding. Qwen advises against greedy decoding in thinking mode (it loops); its recommended settings are
# T=0.6, top_p=0.95, top_k=20 (thinking) and T=0.7, top_p=0.8, top_k=20 (no thinking). Sampling is seeded from the
# prompt, so the same prompt gives the same answer on every run (the repeat check verifies this).
SAMPLING = ({"do_sample": True, "temperature": 0.6, "top_p": 0.95, "top_k": 20} if SETTINGS["enable_thinking"] else
            {"do_sample": True, "temperature": 0.7, "top_p": 0.8, "top_k": 20}) if SETTINGS["sampling"] else \
           {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
_EOS = model_llm.generation_config.eos_token_id if model_llm is not None else tok_llm.eos_token_id
_EOS = [_EOS] if isinstance(_EOS, int) else list(_EOS or [tok_llm.eos_token_id])
THINK_END = tok_llm.convert_tokens_to_ids("</think>") if "</think>" in tok_llm.get_vocab() else None
# Qwen3's own wording for stopping the thinking at a budget (Qwen3 technical report, "thinking budget")
FORCE_ANSWER = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"

def _generate(ids, max_new_tokens: int, eos, seed: int, min_new_tokens: int = 0):
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model_llm.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                                 min_new_tokens=min_new_tokens or None, eos_token_id=eos,
                                 pad_token_id=tok_llm.pad_token_id, **SAMPLING)
    return out[0, ids.shape[1]:]

def _clean(text: str) -> str:
    return re.sub(r"<think>.*?</think>|</?think>", "", text, flags=re.S).strip()

def llm_generate(user_msg: str, max_new_tokens: int, min_new_tokens: int = 0, thinking=None, seed: int = 0):
    """One plain generation (smoke and speed tests). Returns (text, prompt tokens, new tokens)."""
    enc = llm_inputs(user_msg, thinking)
    new = _generate(enc["input_ids"], max_new_tokens, _EOS, seed, min_new_tokens)
    return _clean(tok_llm.decode(new, skip_special_tokens=True)), int(enc["input_ids"].shape[1]), int(len(new))

def llm_answer(user_msg: str, seed: int) -> dict:
    """The answer, in two phases when thinking is on: (1) think until </think> or CONFIG["think_budget"] tokens;
    if the budget runs out, the model is told to stop thinking and answer (Qwen3's budget forcing); (2) the answer,
    with its own budget CONFIG["max_new_tokens"]. The thinking never counts against the answer and never leaks into it."""
    enc = llm_inputs(user_msg)
    ids, n_in = enc["input_ids"], int(enc["input_ids"].shape[1])
    thinking, n_think, forced = "", 0, False
    if SETTINGS["enable_thinking"] and THINK_END is not None:
        new1 = _generate(ids, CONFIG["think_budget"], _EOS + [THINK_END], seed)
        ends = (new1 == THINK_END).nonzero()
        if len(ends):                                         # thinking finished by itself
            cut = int(ends[0]) + 1
            prefix = torch.cat([ids[0], new1[:cut]])
        elif len(new1) and int(new1[-1]) in _EOS:             # answered without closing the thinking block
            text = tok_llm.decode(new1, skip_special_tokens=True)
            return {"answer": _clean(text), "thinking": "", "prompt_tokens": n_in, "think_tokens": 0,
                    "answer_tokens": len(new1), "thinking_forced": False, "hit_limit": False}
        else:                                                 # budget used up: stop thinking, answer now
            cut, forced = len(new1), True
            force = tok_llm(FORCE_ANSWER, add_special_tokens=False, return_tensors="pt")["input_ids"][0].to(ids.device)
            prefix = torch.cat([ids[0], new1, force])
        thinking = _clean(tok_llm.decode(new1[:cut], skip_special_tokens=True))
        n_think, ids = cut, prefix.unsqueeze(0)
    new2 = _generate(ids, CONFIG["max_new_tokens"], _EOS, seed + 1)
    return {"answer": _clean(tok_llm.decode(new2, skip_special_tokens=True)), "thinking": thinking, "prompt_tokens": n_in,
            "think_tokens": n_think, "answer_tokens": int(len(new2)), "thinking_forced": forced,
            "hit_limit": int(len(new2)) >= CONFIG["max_new_tokens"]}

if NEED_LLM:
    _probe = chat_text("x")
    if "qwen3" in SETTINGS["llm_model"].lower():
        print(f"Thinking {'on' if SETTINGS['enable_thinking'] else 'off'} in the prompt:",
              ("<think>\n\n</think>" not in _probe) == SETTINGS["enable_thinking"],
              f"| thinking budget {CONFIG['think_budget']} tokens + answer budget {CONFIG['max_new_tokens']} tokens"
              if SETTINGS["enable_thinking"] else "")
    _ok, _, _ = llm_generate("Reply with one word: the capital of France.", 8, thinking=False)
    if not re.search(r"[A-Za-z]{3,}", _ok) or re.fullmatch(r"(.)\1+", _ok.replace(" ", "")):
        log(f"the answer model's test output looks broken ({_ok!r}); with fp16 this can be a numeric overflow")
    print(f"Answer model test output: {_ok!r} | decoding: " +
          (f"sampling T={SAMPLING['temperature']}, top_p={SAMPLING['top_p']}, top_k={SAMPLING['top_k']}, seeded per prompt"
           if SAMPLING["do_sample"] else "greedy"))
    if SETTINGS["enable_thinking"] and THINK_END is None:
        log("this model's tokenizer has no </think> token: thinking budget not applied")

MODELS = {"embed": SETTINGS["embed_model"], "embed_revision": revision(embed_model, SETTINGS["embed_model"]),
          "reranker": SETTINGS["reranker_model"], "reranker_revision": revision(reranker, SETTINGS["reranker_model"]),
          "llm": SETTINGS["llm_model"], "llm_revision": revision(tok_llm, SETTINGS["llm_model"]),
          "llm_precision": LLM_PRECISION, "enable_thinking": SETTINGS["enable_thinking"],
          "decoding": {k: v for k, v in SAMPLING.items() if v is not None}}
print(f"Loaded: embedder {MODELS['embed']} | reranker {MODELS['reranker']} | answer model "
      f"{MODELS['llm'] + ' (' + LLM_PRECISION + ')' if NEED_LLM else 'not loaded (' + SETTINGS['mode'] + ' mode; tokenizer only)'} | "
      f"GPU in use {gpu_gb()} GB")


In [ ]:
# ================================================================
# 6) Indexes, hybrid search, reranking, context building
# ================================================================
from rank_bm25 import BM25Okapi

def bm25_tokenize(s: str):
    return re.findall(r"[a-z0-9.]+", s.lower())          # keeps "4.7.1" and "excl06" intact

_emb = embed_model.encode([c["embed_text"] for c in final_chunks], batch_size=32, convert_to_numpy=True,
                          show_progress_bar=False).astype("float32")
faiss.normalize_L2(_emb)
index = faiss.IndexFlatIP(_emb.shape[1])
index.add(_emb)
bm25 = BM25Okapi([bm25_tokenize(c["embed_text"]) for c in final_chunks])
IS_PREMIUM = np.array([c["index"] == "premium" for c in final_chunks])

# Premium-table chunks are searched only when the question asks what the policy costs.
_PREMIUM_Q = re.compile(r"\bhow much\b[^?]*\b(cost|premium|pay for (the|this) (policy|plan|cover))|\b(annual|yearly) premium"
                        r"|\bpremium (for|of|amount|rates?|payable for)\b|\bwhat (is|are|would be) the premium"
                        r"|\b(cost|price)s? (of|for)\b|\bcost (more|less)\b|\b(cheaper|expensive)\b", re.I)
_NOT_PREMIUM_Q = re.compile(r"instal+ment|grace period|refund|deduction|income tax|discount|loading", re.I)

def wants_premium(query: str) -> bool:
    return bool(CONFIG["premium_routing"] and N_PREMIUM and _PREMIUM_Q.search(query) and not _NOT_PREMIUM_Q.search(query))

def hybrid_retrieve(query: str):
    """Dense (BGE, query prefix) + BM25 over the policy chunks; for a premium question also over the premium-table
    chunks as a separate index (they are near-identical, so they would lose to policy text in one ranking).
    The union of all top-k lists, ordered by reciprocal rank fusion. Nothing is cut: every candidate is reranked."""
    premium = wants_premium(query)
    q = embed_model.encode([CONFIG["bge_query_prefix"] + query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    _, I = index.search(q, len(final_chunks))
    order_dense = [int(i) for i in I[0] if i != -1]
    order_bm25 = []
    if CONFIG["use_bm25"] and CONFIG["k_bm25"] > 0:
        order_bm25 = [int(i) for i in np.argsort(-bm25.get_scores(bm25_tokenize(query)), kind="stable")]
    policy_only = CONFIG["premium_routing"] and N_PREMIUM
    lists = [[i for i in order_dense if not (policy_only and IS_PREMIUM[i])][:CONFIG["k_dense"]],
             [i for i in order_bm25 if not (policy_only and IS_PREMIUM[i])][:CONFIG["k_bm25"]]]
    if premium:
        lists += [[i for i in order_dense if IS_PREMIUM[i]][:CONFIG["k_premium"]],
                  [i for i in order_bm25 if IS_PREMIUM[i]][:CONFIG["k_premium"]]]
    rrf = {}
    for ranked in lists:
        for rank, i in enumerate(ranked):
            rrf[i] = rrf.get(i, 0.0) + 1.0 / (CONFIG["rrf_k"] + rank)
    return [final_chunks[i] for i in sorted(rrf, key=lambda i: (-rrf[i], i))]

def rerank_all(query: str, docs: List[str], batch_size: int = 8):
    """Cross-encoder score (raw logit, higher = more relevant) of every doc for the query. Docs are batched
    by length (less padding); scores come back in the input order."""
    order = sorted(range(len(docs)), key=lambda j: len(docs[j]))
    scores = np.zeros(len(docs), dtype=np.float32)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        enc = tok_rerank([query] * len(idx), [docs[j] for j in idx], padding=True, truncation=True,
                         max_length=CONFIG["rerank_max_length"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            scores[idx] = reranker(**enc).logits.float().view(-1).cpu().numpy()
    return scores

def rerank_text(c: dict) -> str:
    return c["embed_text"] if CONFIG["rerank_with_label"] else c["text"]

# ---------------------------------------------------------------- context building (v2.3 rules)
def split_sentences(t: str):
    return [p.strip() for p in re.split(r"(?<=[.;])\s+(?=[A-Z0-9·₹(])", t) if p.strip()]

def block_tokens(c: dict) -> int:
    """Tokens a chunk adds to the prompt (answer-model tokenizer): label line + text + separator."""
    if "block_tokens" not in c:
        c["block_tokens"] = tok_len(f"[{c['label']}]") + tok_len(c["text"]) + 2
    return c["block_tokens"]

_TITLE_STOP = {"of", "for", "the", "a", "an", "in", "to", "and", "on", "with", "under", "by", "procedure", "list",
               "section", "notes", "note", "rs", "my"}
_POINTER = re.compile(r"(below|following|as follows|hereunder)\W*$|:\s*$", re.I)

def _words_of(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())

def title_core(title: str) -> frozenset:
    title = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", title)
    return frozenset(w for w in _words_of(title) if w not in _TITLE_STOP)

SECTION_CORES = {sid: title_core(s["section_title"]) for sid, s in SECTIONS.items() if s["index"] == "policy"}

def title_matches(query: str, max_sections: int = 2):
    """Sections whose heading words all appear in the question; a match inside a longer match is dropped."""
    q = set(_words_of(query))
    hits = [sid for sid, core in SECTION_CORES.items() if core and core <= q]
    hits = [a for a in hits if not any(SECTION_CORES[a] < SECTION_CORES[b] for b in hits)]
    hits.sort(key=lambda sid: -len(SECTION_CORES[sid]))
    return hits[:max_sections]

def _parent(sid: str):
    return sid.rsplit(".", 1)[0] if "." in sid else None

def _is_child(a: str, b: str) -> bool:
    return a.startswith(b + ".")

def _same_table_family(a: dict, b: dict) -> bool:
    return a["is_table"] and b["is_table"] and _parent(a["section_id"]) is not None \
        and _parent(a["section_id"]) == _parent(b["section_id"])

def _group_for(i: int):
    """A seed chunk plus the text it depends on: the next chunk of the same unit or a sub-section (once more
    after a "...provided below" pointer), the previous chunk of the same unit or the parent section, and
    the neighbouring year of a plan table. Premium-table chunks are self-describing: no neighbours."""
    seed, out = final_chunks[i], [i]
    if seed["index"] == "premium":
        return out
    if i + 1 < len(final_chunks):
        nxt = final_chunks[i + 1]
        if nxt["index"] == "policy" and (nxt["unit"] == seed["unit"] or _is_child(nxt["section_id"], seed["section_id"])
                                         or _same_table_family(seed, nxt)):
            out.append(i + 1)
            if _POINTER.search(nxt["text"].strip()) and i + 2 < len(final_chunks):
                nn = final_chunks[i + 2]
                if nn["unit"] == nxt["unit"] or _is_child(nn["section_id"], nxt["section_id"]) \
                        or _is_child(nn["section_id"], seed["section_id"]):
                    out.append(i + 2)
    if i > 0:
        prv = final_chunks[i - 1]
        if prv["index"] == "policy" and (prv["unit"] == seed["unit"] or prv["section_id"] == _parent(seed["section_id"])
                                         or _same_table_family(seed, prv)):
            out.append(i - 1)
    return out

def _pinned_group(sid: str):
    """All chunks of a title-matched section, then its sub-sections, in document order."""
    idxs = [c["idx"] for c in final_chunks if c["section_id"] == sid]
    j = idxs[-1] + 1
    while j < len(final_chunks) and _is_child(final_chunks[j]["section_id"], sid):
        idxs.append(j)
        j += 1
    return idxs

def build_context(query: str, candidates: List[dict], scores):
    """Groups of chunk ids (document order inside a group, most relevant group first): title-matched
    sections, then reranked chunks scoring at least the floor, each with its neighbours; hard token cap."""
    budget, floor = CONFIG["context_token_budget"], CONFIG["seed_score_floor"]
    score_of = {c["idx"]: float(s) for c, s in zip(candidates, scores)}
    ranked = sorted(score_of, key=lambda i: (-score_of[i], i))
    plan = [_pinned_group(sid) for sid in title_matches(query)] if CONFIG["title_pinning"] else []
    plan += [_group_for(i) if CONFIG["neighbour_expansion"] else [i] for i in ranked if score_of[i] >= floor]
    used, taken, groups = 0, set(), []
    for g in plan:
        group = []
        for i in g:
            if i in taken:
                continue
            cost = block_tokens(final_chunks[i])
            if used + cost > budget:
                if group or groups:
                    break
                continue
            group.append(i); taken.add(i); used += cost
        if group:
            groups.append(sorted(group))
        if used >= budget - 40:
            break
    return groups, score_of

def merge_group_text(group):
    """Consecutive chunks of one group as one passage; a sentence repeated as overlap is dropped once."""
    parts = []
    for k, i in enumerate(group):
        t = final_chunks[i]["text"]
        if k and group[k - 1] == i - 1 and final_chunks[i - 1]["section_id"] == final_chunks[i]["section_id"]:
            sents, prev = split_sentences(t), split_sentences(final_chunks[i - 1]["text"])
            if sents and prev and sents[0] == prev[-1]:
                t = " ".join(sents[1:])
        parts.append(t)
    return " ".join(p for p in parts if p)

def group_label(group):
    labels = list(dict.fromkeys(final_chunks[i]["label"] for i in group))
    return "[" + " → ".join(labels) + "]"

def format_groups(groups) -> str:
    return "\n\n".join(f"{group_label(g)}\n{merge_group_text(g)}" for g in groups)

print(f"Indexes ready: {index.ntotal} chunks (dense + BM25); premium routing "
      f"{'on' if CONFIG['premium_routing'] and N_PREMIUM else 'off (no premium chunks)'}; "
      f"{sum(len(v) > 0 for v in SECTION_CORES.values())} section headings available for title pinning")


In [ ]:
# ================================================================
# 7) Prompt, cached generation, LangGraph pipeline
# ================================================================
from langgraph.graph import StateGraph, END

NO_ANSWER_TEXT = "Not specified in the provided policy text."
PROMPT = """You are a professional insurance policy analyst.

Use ONLY the information provided in the CONTEXT to answer the QUESTION.
Do NOT add, assume, or infer details beyond the given text.
Provide a clear, factual, and complete answer in full sentences.
When relevant, cite the section name or number from the CONTEXT.
Each CONTEXT passage starts with the section(s) it comes from. A condition, limit or exclusion applies only to the cover named in its own section; do not apply it to a different cover.
Do not present items listed under a different section as part of the cover asked about.
If a passage says a list is provided below, include that list when it appears in the CONTEXT.
When comparing plans, take all figures from the same year's table and name the year.
If the CONTEXT covers only part of the question, answer that part and say what is not shown.
If the answer cannot be found, reply: “Not specified in the provided policy text.”

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

def build_prompt(query: str, context: str) -> str:
    return PROMPT.format(context=context, query=query.strip())

def is_no_answer(ans: str) -> bool:
    return (ans or "").strip().strip('"“”').strip() == NO_ANSWER_TEXT

# ---------------------------------------------------------------- answer cache (keyed by the exact prompt + model settings)
CACHE_FILE = OUT / "answer_cache.jsonl"
CACHE = {}
for _src in [DRIVE / CACHE_FILE.name if DRIVE else None, CACHE_FILE]:
    if _src is not None and Path(_src).exists():
        for _line in Path(_src).read_text(encoding="utf-8").splitlines():
            try:
                _r = json.loads(_line)
                CACHE[_r["key"]] = _r
            except Exception:
                pass
if CACHE:                                      # after a disconnect: the Drive copy and this runtime's copy, merged
    CACHE_FILE.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in CACHE.values()), encoding="utf-8")
# The model revision is recorded with the results (RUN["models"]) but kept out of the key: v3.1 keys hold None, and a
# key that changed with whether the Hub answered the revision lookup would make cached answers miss for no reason.
_GEN_SIG = {"llm": SETTINGS["llm_model"], "revision": None, "precision": MODELS["llm_precision"],
            "thinking": SETTINGS["enable_thinking"], "max_new_tokens": CONFIG["max_new_tokens"],
            "decoding": MODELS["decoding"]}
if SETTINGS["enable_thinking"]:
    _GEN_SIG.update(think_budget=CONFIG["think_budget"], phases="think-then-answer v1")

def answer_key(prompt: str) -> str:
    return hashlib.sha256((json.dumps(_GEN_SIG, sort_keys=True) + "\n" + prompt).encode()).hexdigest()[:24]

GEN_LOG = {}            # prompt key -> generation record used in this session

def generate_answer(prompt: str, use_cache: bool = True) -> str:
    key = answer_key(prompt)
    if use_cache and key in CACHE:
        GEN_LOG[key] = dict(CACHE[key], from_cache=True)
        return CACHE[key]["answer"]
    t0 = time.perf_counter()
    out = llm_answer(prompt, seed=int(key[:8], 16))          # the seed comes from the prompt: same prompt, same answer
    rec = {"key": key, **out, "new_tokens": out["think_tokens"] + out["answer_tokens"],
           "seconds": round(time.perf_counter() - t0, 2), "created": datetime.datetime.now().isoformat(timespec="seconds")}
    ans = out["answer"]
    if use_cache:
        CACHE[key] = rec
        with open(CACHE_FILE, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        mirror(CACHE_FILE)
    GEN_LOG[key] = dict(rec, from_cache=False)
    print(f"    answer: {rec['answer_tokens']} tokens" + (f" after {rec['think_tokens']} thinking tokens"
          + (" (thinking budget reached: told to answer)" if rec["thinking_forced"] else "") if rec["think_tokens"] else "")
          + f" in {rec['seconds']:.0f}s" + (" (answer stopped at the length limit)" if rec["hit_limit"] else ""))
    return ans

# ---------------------------------------------------------------- LangGraph: retrieve -> rerank -> gate -> answer | no_answer
class RAGState(TypedDict, total=False):
    query: str
    retrieved: List[dict]      # search candidates (+ title-matched sections)
    scores: List[float]        # reranker score of each candidate
    groups: List[List[int]]    # context passages: chunk ids, most relevant group first
    top_score: float           # best reranker score (the gate uses it)
    premium: bool              # premium-table chunks were searched
    answer: str
    prompt_key: str
    error: str
    oom_retry: bool

def retrieve_node(state: RAGState):
    try:
        q = state["query"].strip()
        cands = hybrid_retrieve(q)
        if CONFIG["title_pinning"]:                     # pinned sections must be scored too
            have = {c["idx"] for c in cands}
            for sid in title_matches(q):
                cands += [final_chunks[i] for i in _pinned_group(sid) if i not in have]
                have.update(_pinned_group(sid))
        return {"retrieved": cands, "premium": wants_premium(q)}
    except Exception as e:
        traceback.print_exc()
        return {"retrieved": [], "error": f"retrieve: {type(e).__name__}: {e}"}

def rerank_node(state: RAGState):
    empty = {"scores": [], "groups": [], "top_score": float("-inf")}
    try:
        cands = state.get("retrieved") or []
        if not cands:
            return empty
        scores = rerank_all(state["query"], [rerank_text(c) for c in cands])
        groups, _ = build_context(state["query"], cands, scores)
        return {"scores": [float(s) for s in scores], "groups": groups, "top_score": float(scores.max())}
    except Exception as e:
        traceback.print_exc()
        return {**empty, "error": f"rerank: {type(e).__name__}: {e}"}

def gate(state: RAGState) -> str:
    return "answer" if state.get("groups") and state.get("top_score", float("-inf")) >= CONFIG["rerank_score_threshold"] \
        else "no_answer"

def _shorten(groups, limit):
    keep, used = [], 0
    for g in groups:
        part = []
        for i in g:
            t = block_tokens(final_chunks[i])
            if used + t > limit and (keep or part):
                break
            part.append(i); used += t
        if part:
            keep.append(part)
        if used >= limit:
            break
    return keep

def answer_node(state: RAGState):
    try:
        prompt = build_prompt(state["query"], format_groups(state["groups"]))
        try:
            return {"answer": generate_answer(prompt), "prompt_key": answer_key(prompt)}
        except torch.cuda.OutOfMemoryError:
            free_gpu()
            keep = _shorten(state["groups"], CONFIG["oom_retry_context_tokens"])
            log(f"out of GPU memory for {state['query'][:50]!r}: retried with {sum(map(len, keep))} chunks")
            prompt = build_prompt(state["query"], format_groups(keep))
            return {"answer": generate_answer(prompt), "prompt_key": answer_key(prompt), "groups": keep, "oom_retry": True}
    except Exception as e:
        traceback.print_exc()
        return {"answer": NO_ANSWER_TEXT, "error": f"answer: {type(e).__name__}: {e}"}
    finally:
        free_gpu()

def no_answer_node(state: RAGState):
    return {"answer": NO_ANSWER_TEXT}

_g = StateGraph(RAGState)
for _name, _fn in [("retrieve", retrieve_node), ("rerank", rerank_node), ("answer", answer_node), ("no_answer", no_answer_node)]:
    _g.add_node(_name, _fn)
_g.set_entry_point("retrieve")
_g.add_edge("retrieve", "rerank")
_g.add_conditional_edges("rerank", gate, {"answer": "answer", "no_answer": "no_answer"})
_g.add_edge("answer", END)
_g.add_edge("no_answer", END)
app = _g.compile()

def ask(question: str):
    """One question through the pipeline, with its sources (costs one answer-model call unless cached)."""
    r = app.invoke({"query": question})
    for g in r.get("groups", []):
        print("  source:", group_label(g)[:120])
    print("\nQ:", question, "\nA:", r["answer"])
    return r

print(f"Pipeline ready: retrieve -> rerank -> gate (best score >= {CONFIG['rerank_score_threshold']}) -> answer | no_answer"
      f" | answer cache: {len(CACHE)} answers")


## Evaluation

In [ ]:
# ================================================================
# 8) Checks and metrics (deterministic; no LLM judge)
# ================================================================
# Per question:  fully correct (PRIMARY) = every key fact, no critical error, nothing missing or cut off;
#                for a not-answerable question: says so at the start, with no critical error.
# Critical errors: wrong plan-table value or premium row, known false statement, another cover's rule,
#                  wrong yes/no, wrong order, a figure that is not in the context, answering a not-answerable question.
# Retrieval (from the evidence passages): search recall, context recall, MRR@10, context precision.
# Each problem is tagged with the pipeline stage where it started: corpus, retrieval, context, gate, generation.

# ---------------------------------------------------------------- figures and section citations
_WORD_NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
             "ten": 10, "twelve": 12, "fifteen": 15, "twenty": 20, "thirty": 30, "forty five": 45, "sixty": 60, "ninety": 90}
_UNIT = {"day": "days", "month": "months", "year": "years", "hour": "hours"}

def figures(text: str) -> set:
    """Amounts (lakh/crore, rupee and comma-grouped forms unified), percentages and durations."""
    t = (text or "").lower().replace("₹", " ₹ ").replace("-", " ")
    for w, n in sorted(_WORD_NUM.items(), key=lambda x: -len(x[0])):
        t = re.sub(rf"\b{w}\b", str(n), t)
    out = set()
    for m in re.finditer(r"(?:₹|\brs\.?|\binr)?\s*(\d+(?:\.\d+)?)\s*(lakhs?|lacs?|crores?|cr)\b", t):
        out.add(("inr", int(round(float(m.group(1)) * (1e5 if m.group(2).startswith("la") else 1e7)))))
    for m in re.finditer(r"(?:₹|\brs\.?|\binr)\s*(\d[\d,]*(?:\.\d+)?)(?!\d|[.,]\d)(?!\s*(?:lakhs?|lacs?|crores?|cr)\b)", t):
        out.add(("inr", int(round(float(m.group(1).replace(",", ""))))))
    for m in re.finditer(r"(?<![\d.,])(\d{1,3}(?:,\d{2,3})+)(?![\d,])(?!\s*(?:lakhs?|lacs?|crores?|cr)\b)", t):
        out.add(("inr", int(m.group(1).replace(",", ""))))           # premium tables: "Age 40: 10,300 / 13,300"
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*%", t):
        out.add(("pct", float(m.group(1))))
    for m in re.finditer(r"(\d+)\s*(?:(?:consecutive|continuous|calendar|policy|completed|full)\s+){0,2}(day|month|year|hour)s?\b", t):
        out.add((_UNIT[m.group(2)], int(m.group(1))))
    return out

def _plain_numbers(text: str) -> set:
    return {float(x.replace(",", "")) for x in re.findall(r"(?<![\d.])\d[\d,]*(?:\.\d+)?", text or "")}

def unsupported_figures(answer: str, context: str, question: str = "", derived=()) -> list:
    """Figures in the answer that are not in the context or the question. An amount also counts as supported when
    the context has the same number without a currency mark (tables write "max upto 6000")."""
    extra, plain = {tuple(d) for d in derived}, _plain_numbers(context) | _plain_numbers(question)
    left = {(k, v) for k, v in figures(answer) - figures(context) - figures(question) - extra
            if not (k == "inr" and float(v) in plain)}
    return sorted(f"Rs {v:,.0f}" if k == "inr" else f"{v:g}%" if k == "pct" else f"{v:g} {k}" for k, v in left)

def cited_sections(text: str) -> set:
    ids = set()
    for m in re.finditer(r"sections?\s+((?:\d+(?:\.\d+)*(?:\.[a-z])?(?:\s*(?:,|and|&)\s*)?)+)", (text or "").lower()):
        ids.update(s.rstrip(".") for s in re.findall(r"\d+(?:\.\d+)*", m.group(1)))
    return ids

def unsupported_citations(answer: str, context: str) -> list:
    ctx_ids = set(re.findall(r"\d+(?:\.\d+)+|\b\d+\b", context or ""))
    return sorted(s for s in cited_sections(answer) if s not in ctx_ids)

# Plan illustration (Section 2.1): Base Sum Insured Rs. 10 lakh, no claims; values in lakh
PLAN_TABLE = {}
for _plan, _single, _all, _sb, _plus in [("suraksha", (10, 11, 12), (20, 21, 22), 0, (0, 1, 2)),
                                         ("secure", (20, 25, 30), (30, 35, 40), 10, (0, 5, 10)),
                                         ("super", (30, 35, 40), (40, 45, 50), 20, (0, 5, 10))]:
    for _y in (1, 2, 3):
        PLAN_TABLE.update({(_plan, "single", _y): _single[_y - 1], (_plan, "all", _y): _all[_y - 1],
                           (_plan, "secure_benefit", _y): _sb, (_plan, "plus", _y): _plus[_y - 1],
                           (_plan, "base", _y): 10, (_plan, "restore", _y): 10})

# ---------------------------------------------------------------- plan-table values
_PLAN_RES = [("global_plus", r"(?:optima\s+)?secure\s+global\s+plus|global\s+plus"),
             ("global", r"(?:optima\s+)?secure\s+global|\bglobal\b(?!\s+(?:health|cover))"),
             ("super", r"(?:optima\s+)?super[\s-]*secure"),
             ("suraksha", r"(?:optima\s+)?suraksha"),
             ("secure", r"optima\s+secure(?!\s+(?:global|benefit|policy))|\bsecure\s+plan|(?<=for )secure\b(?!\s+benefit)"
                        r"|(?<=for the )secure\b(?!\s+benefit)|(?<=, )secure\b(?!\s+benefit)|(?<=and )secure\b(?!\s+benefit)"
                        r"|(?m:^[ \t\-*•]*(?:\d+\.\s*)?secure\b(?!\s+benefit))")]
_ROW_RES = [("secure_benefit", r"secure\s+benefit"), ("plus", r"plus\s+benefit|cumulative\s+bonus"),
            ("restore", r"restore"), ("base", r"base\s+sum\s+insured"),
            ("single", r"single[\s-]+(?:hospitali[sz]ation[\s-]+)?claim|per[\s-]+claim|any\s+one\s+claim|one\s+claim"),
            ("all", r"all\s+(?:the\s+)?(?:hospitali[sz]ation\s+)?claims|aggregate|total\s+(?:of\s+all\s+)?claims"
                    r"|(?:annual|yearly)\s+(?:limit|maximum)")]
_YEAR_RES = [("all", r"\b(?:in\s+)?(?:all|each\s+of\s+the)\s+(?:three\s+|3\s+)?(?:policy\s+)?years\b|\bacross\s+(?:all\s+)?(?:the\s+)?(?:three\s+)?years\b"),
             ("num", r"\byear\s*[-‐]?\s*(1|2|3|one|two|three)\b|\b(first|second|third|1st|2nd|3rd)\s+(?:policy\s+)?year\b")]
_YEAR_WORD = {"1": 1, "one": 1, "first": 1, "1st": 1, "2": 2, "two": 2, "second": 2, "2nd": 2,
              "3": 3, "three": 3, "third": 3, "3rd": 3}
_AMOUNT_RES = [r"(?:(?:₹|\brs\.?|\binr)\s*)?(\d+(?:\.\d+)?)\s*(lakhs?|lacs?|crores?)\b",
               r"(?:₹|\brs\.?|\binr)\s*(\d{1,3}(?:,\d{2,3}){2,}|\d{6,})(?![\d,])"]
_PLAN_NAME = {"suraksha": "Optima Suraksha", "secure": "Optima Secure", "super": "Optima Super Secure",
              "global": "Optima Secure Global", "global_plus": "Optima Secure Global Plus"}
_ROW_NAME = {"single": "single claim", "all": "all claims in a year", "secure_benefit": "Secure Benefit",
             "plus": "Plus Benefit / Cumulative Bonus", "base": "Base Sum Insured", "restore": "Restore Benefit"}

def _events(text: str):
    """Plan, row, year and amount mentions with their positions (longer plan names masked first)."""
    ev, work = [], text
    for name, pat in _PLAN_RES:
        for m in re.finditer(pat, work, re.I):
            ev.append((m.start(), m.end(), "plan", name))
        work = re.sub(pat, lambda m: " " * len(m.group(0)), work, flags=re.I)
    for name, pat in _ROW_RES:
        ev += [(m.start(), m.end(), "row", name) for m in re.finditer(pat, text, re.I)]
    for kind, pat in _YEAR_RES:
        for m in re.finditer(pat, text, re.I):
            if kind == "all":
                ev.append((m.start(), m.end(), "year", "all"))
            else:
                ev.append((m.start(), m.end(), "year", _YEAR_WORD[(m.group(1) or m.group(2)).lower()]))
    taken = []
    for k, pat in enumerate(_AMOUNT_RES):
        for m in re.finditer(pat, text, re.I):
            if any(a <= m.start() < b for a, b in taken):
                continue
            num = float(m.group(1).replace(",", ""))
            val = (num * 100 if m.group(2).lower().startswith("cr") else num) if k == 0 else num / 1e5
            taken.append((m.start(), m.end()))
            ev.append((m.start(), m.end(), "amount", round(val, 3)))
    return sorted(ev)

def _first(text: str, kind: str):
    return next((v for _, _, k, v in _events(text) if k == kind), None)

def _prose_claims(text: str, state: dict):
    """Claims (plan, row, year, lakh) from running text; state carries plan/row/year headings."""
    ev = _events(text)
    breaks = [m.end() for m in re.finditer(r"\n|;|(?<=[.!?])\s+", text)]
    seg_of = lambda pos: sum(b <= pos for b in breaks)
    amounts = [e for e in ev if e[2] == "amount"]
    claims = []
    seg_has_amount = {seg_of(a[0]) for a in amounts}
    headings = [(x[0], x[3]) for x in ev if x[2] == "year" and seg_of(x[0]) not in seg_has_amount]   # "Year 2:" lines
    for n, (s, e, _, val) in enumerate(amounts):
        nxt = amounts[n + 1][0] if n + 1 < len(amounts) else len(text)
        if re.match(r"\s*(?:more|extra|additional|higher|lower|less|greater)\b", text[e:e + 20], re.I) or re.search(
                r"\b(?:additional|extra|difference of|more than|less than|by|increase of)\s*(?:a\s+)?(?:₹|rs\.?|inr)?\s*$",
                text[max(0, s - 30):s], re.I):
            continue                                   # a difference ("Rs. 10 lakh more"), not a table value
        seg_end = min([b for b in breaks if b > e] + [len(text)])
        stop = min(nxt, seg_end)
        after = [x for x in ev if e <= x[0] < stop and x[2] != "amount"]
        lead = re.match(r"[\s,(]*(?:for|under|in|of)\s+(?:the\s+)?", text[e:stop], re.I)
        # plan: "for <plan>[, <plan> and <plan>]" right after the amount, else the latest plan before it
        plans = []
        if lead:
            pos = e + lead.end()
            for x in after:
                if x[2] == "plan" and x[0] <= pos + 2:
                    plans.append(x[3])
                    tail = re.match(r"\s*(?:,|and|&)?\s*(?:the\s+)?", text[x[1]:stop])
                    pos = x[1] + (tail.end() if tail else 0)
        if not plans:
            before = [x for x in ev if x[2] == "plan" and x[1] <= s]
            plans = [before[-1][3]] if before else ([state["plan"]] if state.get("plan") else [])
        # row: "for a single claim" right after the amount, else the latest row mention before it
        row = None
        rlead = re.match(r"\s*(?:for|per|on|in)\s+(?:a\s+|the\s+|any\s+|each\s+)?", text[e:stop], re.I)
        if rlead:
            row = next((x[3] for x in after if x[2] == "row" and x[0] <= e + rlead.end() + 1), None)
        if row is None:                          # same line first, then a heading line ("Secure Benefit:") above it
            line_start = text.rfind("\n", 0, s) + 1
            same_line = [x for x in ev if x[2] == "row" and line_start <= x[0] and x[1] <= s]
            head_rows = [x for x in ev if x[2] == "row" and x[1] <= s and not any(
                a[0] >= text.rfind("\n", 0, x[0]) + 1 and a[0] < (text.find("\n", x[0]) % (len(text) + 1)) for a in amounts)]
            row = same_line[-1][3] if same_line else (head_rows[-1][3] if head_rows else state.get("row"))
        # year: "in Year 2" / "(Year 2)" / "in all three years" right after, else earlier in the sentence,
        # else a heading such as "Year 2:"
        year = None
        ylead = re.match(r"[\s,(\-]*(?:in|for|during|of|from)?\s*(?:the\s+)?", text[e:stop], re.I)
        for x in after:
            if x[2] == "year" and x[0] <= e + ylead.end() + 1 and not re.match(r"\s*[:\-–|]", text[x[1]:x[1] + 3]):
                year = x[3]
                break
        if year is None:
            same = [x for x in ev if x[2] == "year" and x[1] <= s and seg_of(x[0]) == seg_of(s)]
            heads = [y for pos, y in headings if pos < s]
            year = same[-1][3] if same else (heads[-1] if heads else state.get("heading_year"))
        claims += [(p, row, year, val) for p in plans]
    plans_seen = [x for x in ev if x[2] == "plan"]
    rows_seen = [x for x in ev if x[2] == "row"]
    if plans_seen:
        state["plan"] = plans_seen[-1][3]
    if rows_seen:
        state["row"] = rows_seen[-1][3]
    if headings:
        state["heading_year"] = headings[-1][1]
    return claims

def _markdown_table_claims(lines, state: dict):
    rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in lines]
    rows = [r for r in rows if not all(re.fullmatch(r":?-{2,}:?", c) for c in r if c)]
    if len(rows) < 2:
        return []
    header, claims = rows[0], []
    cols = [{"plan": _first(c, "plan"), "row": _first(c, "row"), "year": _first(c, "year")} for c in header]
    for r in rows[1:]:
        label = {"plan": _first(r[0], "plan"), "row": _first(r[0], "row"), "year": _first(r[0], "year")} if r else {}
        for j, cell in enumerate(r[1:], start=1):
            col = cols[j] if j < len(cols) else {}
            pick = lambda k: label.get(k) or col.get(k) or state.get("heading_year" if k == "year" else k)
            for _, _, kind, val in _events(cell):
                if kind == "amount" and pick("plan"):
                    claims.append((pick("plan"), pick("row"), pick("year"), val))
    return claims

def question_state(question: str) -> dict:
    """Defaults taken from the question for answers that do not repeat them ("Rs. 45 lakh."): its plan when it
    names exactly one, its row, and its year when it names one year."""
    ev = _events(question or "")
    plans = {v for _, _, k, v in ev if k == "plan"}
    years = [v for _, _, k, v in ev if k == "year" and v != "all"]
    st = {"row": _first(question or "", "row")} if _first(question or "", "row") else {}
    if len(plans) == 1:
        st["plan"] = next(iter(plans))
    if len(set(years)) == 1:
        st["heading_year"] = years[0]
    return st

def parse_table_claims(answer: str, question: str = ""):
    """All (plan, row, year, value in lakh) statements in an answer: running text and markdown tables."""
    state, claims, block, table = question_state(question), [], [], []
    for line in (answer or "").splitlines() + [""]:
        if line.strip().startswith("|"):
            if block:
                claims += _prose_claims("\n".join(block), state)
                block = []
            table.append(line)
        else:
            if table:
                claims += _markdown_table_claims(table, state)
                table = []
            block.append(line)
    if block:
        claims += _prose_claims("\n".join(block), state)
    return claims

def table_check(answer: str, cells, question: str = "") -> dict:
    """Compare stated plan-table values with Sections 2.1-2.3. A value stated without a year counts
    as correct if it matches that plan and row in any year (reported as 'year not named')."""
    expected = {tuple(c) for c in cells}
    correct, wrong_cells, wrong, unnamed = set(), set(), [], 0
    for plan, row, year, val in parse_table_claims(answer, question):
        if plan not in ("suraksha", "secure", "super") or row is None:
            continue
        truth = {y: PLAN_TABLE.get((plan, row, y)) for y in (1, 2, 3)}
        if year is None:
            ys = [y for y, t in truth.items() if t is not None and abs(t - val) < 1e-6]
            if ys:
                unnamed += 1
                correct.update((plan, row, y) for y in ys)
            else:
                wrong.append(f"{_PLAN_NAME[plan]}, {_ROW_NAME[row]} (year not named): answer {val:g} lakh, "
                             f"policy {'/'.join(f'{t:g}' for t in truth.values())} lakh for years 1/2/3")
            continue
        for y in ((1, 2, 3) if year == "all" else (year,)):
            t = truth.get(y)
            if t is None:
                continue
            if abs(t - val) < 1e-6:
                correct.add((plan, row, y))
            else:
                wrong_cells.add((plan, row, y))
                wrong.append(f"{_PLAN_NAME[plan]}, {_ROW_NAME[row]}, Year {y}: answer {val:g} lakh, policy {t:g} lakh")
    return {"expected": len(expected), "correct": len(expected & correct), "wrong": wrong,
            "missing": [f"{_PLAN_NAME[p]}, {_ROW_NAME[r]}, Year {y}" for p, r, y in sorted(expected - correct - wrong_cells)],
            "year_not_named": unnamed}

# ---------------------------------------------------------------- scope, false statements, polarity

# ---------------------------------------------------------------- refusals, admissions, yes/no
_DECLINE = re.compile(
    r"not (specified|mentioned|stated|provided|included|covered|listed|available|given|shown|found|offered|described|"
    r"defined|addressed|part of|applicable|contain)|(does|do|did|is|are)( not|n[’']t) (specify|mention|cover|state|provide|"
    r"include|contain|list|offer|have|say|describe|address|apply|exist|cover|appear|explicitly)|"
    r"\bno (information|mention|details|reference|data|such|specific|indication|provision)\b|\bthere (is|are) no\b|"
    r"cannot (be )?(found|determined|answered|confirmed|provide)|unable to (find|determine|answer|confirm|provide)|"
    r"\bnot (possible|able) to (determine|answer|say)|outside the scope|beyond the (scope|provided)", re.I)

def sentences_of(text: str):
    return [s.strip() for s in re.split(r"\n+|(?<=[.!?])\s+", text or "") if s.strip()]

def declines_at_start(answer: str) -> bool:
    """A refusal or "the policy does not say" within the first two sentences (the stricter rule: an answer
    that invents first and hedges at the end does not count)."""
    if is_no_answer(answer):
        return True
    head = " ".join(re.sub(r"^[\W_]+", "", s) for s in sentences_of(answer)[:2])
    return bool(_DECLINE.search(head))

def admits_gap(answer: str) -> bool:
    return bool(_DECLINE.search(answer or "")) or bool(re.search(
        r"\bno\b[^.;:\n]{0,60}\b(mention|mentioned|information|specified|stated|provided|details|listed)\b", answer or "", re.I))

def answer_polarity(answer: str):
    """'yes' / 'no' when the answer opens with one (e.g. "No, return transportation is not covered")."""
    m = re.match(r"[\W_]*(?:(?:based on|according to|as per)[^,\n]{0,80},\s*)?(yes|no)\b\s*[,.!:;–—-]", (answer or "").lower())
    return m.group(1) if m else None

# ---------------------------------------------------------------- scope: rules of another cover used
def context_blocks(context: str):
    """(section ids, text) for each passage of a formatted context."""
    out = []
    for b in re.split(r"\n\n(?=\[)", context or ""):
        head, _, body = b.partition("\n")
        out.append((re.findall(r"Section ([\w.\-]+?)(?: -|\]| →|$)", head), body))
    return out

def _bigrams(text: str) -> set:
    w = re.findall(r"[a-z0-9]+", norm_text(text))
    return set(zip(w, w[1:]))

def _in_sections(sid: str, prefixes) -> bool:
    return any(sid == p or sid.startswith(p + ".") for p in prefixes)

def off_scope_hits(answer: str, context: str, prefixes):
    """Sections of another cover or procedure that the answer cites, or copies a sentence from."""
    if not prefixes or not answer or is_no_answer(answer):
        return []
    hits = [f"cites Section {s}" for s in sorted(cited_sections(answer)) if _in_sections(s, prefixes)]
    blocks = [(ids, _bigrams(body)) for ids, body in context_blocks(context)]
    for s in sentences_of(answer):
        sb = _bigrams(s)
        if len(sb) < 5 or not blocks:
            continue
        ids, bg = max(blocks, key=lambda b: len(sb & b[1]))
        in_scope = max((len(sb & b) for i2, b in blocks if not any(_in_sections(i, prefixes) for i in i2)), default=0)
        if len(sb & bg) / len(sb) >= 0.5 and any(_in_sections(i, prefixes) for i in ids) \
                and not any(not _in_sections(i, prefixes) for i in ids) and in_scope / len(sb) < 0.5:
            hits.append(f"copies Section {ids[0]} (\"{s[:50]}…\")")    # not when the right section has it too
    return hits

def cut_off(answer: str, gen):
    if gen and gen.get("hit_limit") is not None:
        return bool(gen["hit_limit"])
    a = (answer or "").rstrip()
    return bool(a) and not is_no_answer(a) and not re.search(r"[.!?:)\]\"”'*|]$", a)

# ---------------------------------------------------------------- one verdict per answer
STAGES = ["reliability", "corpus", "retrieval", "context", "gate", "generation"]
CRITICAL = {"table_wrong", "known_false", "off_scope", "polarity", "wrong_order", "unsupported_figures", "not_declined"}
INCOMPLETE = {"missing_fact", "table_missing", "refused", "no_admit", "cut_off"}

def score_answer(item: dict, answer: str, context: str, gen=None, gate_route="answer", error=None,
                 cand_norm=None) -> dict:
    """Verdict for one answer. cand_norm = normalised text of all search candidates (live runs only), used to
    tell a search miss from a context miss."""
    issues = []
    add = lambda stage, kind, text: issues.append({"stage": stage, "kind": kind, "issue": text})
    a, a_norm, c_norm = answer or "", norm_text(answer), norm_text(context)
    v = {"refused": is_no_answer(a), "fact_recall": None, "facts_missing": [], "table": None}
    if error:
        add("reliability", "error", f"pipeline error: {str(error)[:80]}")
    if item["answerable"]:
        found = [fact_found(f, a_norm) for f in item["facts"]]
        v["fact_recall"] = sum(found) / len(found) if found else None
        v["facts_missing"] = [fact_label(f) for f, ok in zip(item["facts"], found) if not ok]
        v["false_refusal"] = v["refused"] or (declines_at_start(a) and not any(found))
        for f, ok in zip(item["facts"], found):
            if ok:
                continue
            derived = any(fact_found(f, str(x[1])) for x in item.get("derived_figures", []))
            if not derived and not fact_found(f, _CORPUS_NORM):
                add("corpus", "missing_fact", f"not in the chunk file: {fact_label(f)}")
            elif not derived and cand_norm is not None and not fact_found(f, cand_norm):
                add("retrieval", "missing_fact", f"search did not find: {fact_label(f)}")
            elif not derived and not fact_found(f, c_norm):
                add("context", "missing_fact", f"not in the context: {fact_label(f)}")
            elif not v["refused"] and gate_route == "answer":
                add("generation", "missing_fact", f"in the context, not in the answer: {fact_label(f)}")
        if gate_route == "no_answer":
            add("gate", "refused", "refused by the relevance gate")
        elif v["false_refusal"]:
            add("generation", "refused", "model refused")
        if not v["refused"] and a:
            if item.get("table_cells"):
                v["table"] = tc = table_check(a, item["table_cells"], item["question"])
                if tc["wrong"]:
                    add("generation", "table_wrong", f"wrong table values ({len(tc['wrong'])}): " + "; ".join(tc["wrong"][:3]))
                if tc["missing"]:
                    add("generation", "table_missing", f"table values missing: {len(tc['missing'])} of {tc['expected']}")
            if item.get("ordered"):
                pos = [fact_pos(f, a_norm) for f in item["facts"]]
                pos = [p for p in pos if p is not None]
                if pos != sorted(pos):
                    add("generation", "wrong_order", "key facts in the wrong order")
            if item.get("polarity") and answer_polarity(a) and answer_polarity(a) != item["polarity"]:
                add("generation", "polarity", f"answers '{answer_polarity(a)}', the policy says '{item['polarity']}'")
            if item.get("expect_admit") and not admits_gap(a):
                add("generation", "no_admit", "does not say which part the policy does not answer")
            if cut_off(a, gen):
                add("generation", "cut_off", "stopped at the answer length limit")
            missing_said = _DECLINE.search(a) and [fact_label(f) for f, ok in zip(item["facts"], found)
                                                    if not ok and fact_found(f, c_norm)]
            if missing_said:
                add("generation", "says_missing", "says something is not shown, but the context has: " + ", ".join(missing_said))
    elif not declines_at_start(a):
        add("generation", "not_declined", "answered instead of saying the policy does not cover it")
    if a and not is_no_answer(a):
        for p in item.get("must_not", []):
            m = re.search(p, a_norm)
            if m:
                add("generation", "known_false", f"known false or wrong-row statement: \"{m.group(0)[:60]}\"")
        off = off_scope_hits(a, context, item.get("off_scope"))
        if off:
            add("generation", "off_scope", "uses another cover's or procedure's section: " + "; ".join(off[:2]))
        uf = unsupported_figures(a, context, item["question"], item.get("derived_figures", []))
        if uf:
            add("generation", "unsupported_figures", "figures not in the context: " + ", ".join(uf))
        uc = unsupported_citations(a, context)
        if uc:
            add("generation", "unsupported_citations", "cites section numbers not in the context: " + ", ".join(uc))
    kinds = {i["kind"] for i in issues}
    v["critical"] = sorted(kinds & CRITICAL)
    if item["answerable"]:
        v["fully_correct"] = not error and not (kinds & (CRITICAL | INCOMPLETE))
    else:
        v["abstained"] = declines_at_start(a)
        v["fully_correct"] = not error and v["abstained"] and not (kinds & CRITICAL)
    v["issues"] = issues
    v["first_stage"] = None if v["fully_correct"] else next((s for s in STAGES if any(i["stage"] == s for i in issues)), None)
    return v

# ---------------------------------------------------------------- retrieval metrics from the evidence passages
def _words_of_ids(ids):
    return " ".join(final_chunks[i]["words"] for i in sorted(set(ids)))

def retrieval_metrics(item: dict, cand_ids, ranked_ids, ctx_ids) -> dict:
    ev = item["evidence"]
    if not item["answerable"] or not ev:
        return {}
    relevant = lambda i: any(anchor_in(e, final_chunks[i]["words"]) for e in ev)
    cand_w, ctx_w = _words_of_ids(cand_ids), _words_of_ids(ctx_ids)
    rr = next((1.0 / r for r, i in enumerate(ranked_ids[:10], start=1) if relevant(i)), 0.0)
    tok = sum(block_tokens(final_chunks[i]) for i in ctx_ids)
    rel_tok = sum(block_tokens(final_chunks[i]) for i in ctx_ids if relevant(i))
    ctx_norm = norm_text(" ".join(final_chunks[i]["label"] + " " + final_chunks[i]["text"] for i in ctx_ids))
    return {"evidence_in_chunk_file": float(np.mean([anchor_in(e, _CORPUS_WORDS) for e in ev])),
            "search_recall": float(np.mean([anchor_in(e, cand_w) for e in ev])),
            "context_recall": float(np.mean([anchor_in(e, ctx_w) for e in ev])),
            "mrr10": rr, "context_precision": rel_tok / tok if tok else 0.0,
            "facts_in_context": float(np.mean([fact_found(f, ctx_norm) for f in item["facts"]])) if item["facts"] else None}

print(f"Checks ready: {len(CRITICAL)} critical-error kinds, {len(INCOMPLETE)} incompleteness kinds, stages {' > '.join(STAGES)}")


In [ ]:
# ================================================================
# 8b) Hard questions (optional): 15 questions beyond the test set, each shown with the expected answer
# ================================================================
# Not scored automatically and not part of any decision. To use: SETTINGS["hard_questions"] = True, click into
# cell 9 and choose Runtime > Run before (about 25 min with thinking on). Saves hard_questions.csv.
HARD = [
    ("I have Optima Secure with a Rs. 5 lakh Base Sum Insured, in my second policy year with no break. This year I make "
     "three claims: Rs. 4 lakh, then Rs. 12 lakh, then Rs. 4 lakh. How much is paid for each?",
     "Rs 4 lakh (Base); Rs 12 lakh (Base 1 + Plus 2.5 + Secure 5 + Restore 3.5); Rs 1.5 lakh (rest of Restore). Sections 13.2, 4.4-4.6"),
    ("We chose a Rs. 1 lakh Aggregate Deductible. In one year my wife claims Rs. 60,000 and I claim Rs. 80,000. How much "
     "does the insurer pay on a family floater, and how much on an individual policy?",
     "Floater: Rs 40,000 (deductible used up once across the family). Individual: nothing (each person bears up to Rs 1 lakh). Section 4.7"),
    ("On Optima Super Secure I stayed 5 days in a shared room at a network hospital, 1 of those days in the ICU. "
     "How much daily cash do I get?",
     "4 days x Rs 1,000 = Rs 4,000 (no ICU days; cap Rs 6,000; does not reduce the Sum Insured). Section 4.2, Annexure A"),
    ("We are a family of four on a Rs. 15 lakh family floater. How much health check-up cover do we get per year, "
     "and would four individual covers give more?",
     "Floater: Rs 8,000 in total. Individual: Rs 4,000 each = Rs 16,000. Only on continuous renewal, no carry-forward. Section 5"),
    ("My father was discharged on 1 March. His post-hospitalization treatment ended on 20 August, and we submit the bills "
     "on 30 August. Will they be paid?",
     "Yes: the 180-day window runs to about 28 Aug; the deadline is 15 days after treatment ends (4 Sep). Sections 3.6, 11.6"),
    ("The insurer received our last document on 1 June, investigated the claim, and paid on 25 July. Do we get interest?",
     "Yes: with an investigation the limit is 45 days (16 July); interest at 2% above bank rate from 1 June to 25 July. Section 12.3"),
    ("My first policy is 10 months old. I need a knee replacement after a road accident. Is it covered?",
     "Yes: the 24-month waiting period for joint replacement does not apply to accidents. Excl02"),
    ("I declared knee osteoarthritis when I bought the policy. After 30 months, can I claim a knee replacement?",
     "Not yet: pre-existing (36 months) and on the 24-month list; the longer period (36 months) applies. Excl01, Excl02 (iii)"),
    ("After an emergency surgery in Singapore paid under my global cover, I do 2 months of physiotherapy in India. "
     "Is it paid on Optima Secure Global? On Global Plus?",
     "No on both: Global excludes pre/post for an overseas claim even in India; Global Plus pays only pre/post incurred "
     "and paid overseas. Sections 4.9 B(ii), 4.10 (viii)"),
    ("Optima Secure Global, Rs. 1 crore, first year. An emergency in Dubai costs Rs. 1.6 crore. Can the Secure Benefit "
     "or Restore cover the part above Rs. 1 crore?",
     "No: Secure and Restore apply only in India; the global cover pays at most Base SI + Plus (none in year 1), less a "
     "Rs 10,000 per-claim deductible. Section 2.1 note, 4.9"),
    ("The hospital billed laundry charges and mineral water. Does Optima Suraksha pay them? Does Optima Secure?",
     "Suraksha: no (Annexure B non-medical items, no Protect Benefit). Secure: yes, under Protect Benefit for an admissible "
     "Section 3 claim. Sections 4.3, 10.2(k), Annexures A and B"),
    ("I have been insured for 9 continuous years and raised my Sum Insured from Rs. 10 to 20 lakh 3 years ago. "
     "Can the insurer contest a claim for non-disclosure?",
     "Not on the original Rs 10 lakh (8-year moratorium done; only proven fraud or permanent exclusions). The extra "
     "Rs 10 lakh is still contestable (8 years from the increase). Section 12.6"),
    ("On Optima Super Secure, my doctor abroad advised 6 days in hospital and an attendant. Can I claim my wife's flights "
     "and hotel under Overseas Travel Secure?",
     "No: Overseas Travel Secure exists only with Secure Global / Global Plus; Super Secure has no overseas cover. "
     "Section 4.11, Annexure A"),
    ("I ported my policy from another insurer last week. Can I return it in the free look period?",
     "No: free look is for new policies only, not porting or renewal; you can cancel with 15 days' notice for a pro-rata "
     "refund if no claim was admitted. Sections 12.8, 12.12"),
    ("What is the sub-limit for AYUSH treatment on Optima Secure?",
     "Not stated in this document; the sub-limit is in the Policy Schedule. Any figure is invented. Section 3.4"),
]

_hard = HARD if SETTINGS["hard_questions"] and SETTINGS["mode"] == "full" else []
if _hard:
    print(f"Chunk file: {CHUNK_PATH.name}" + ("" if N_PREMIUM else "  <-- old text file: use chunks_from_pdf_v27.json"))
rows = []
for n, (q, expected) in enumerate(_hard, 1):
    print(f"\n{'=' * 110}\n[{n}/{len(HARD)}]")
    t0 = time.perf_counter()
    r = ask(q)
    g = GEN_LOG.get(r.get("prompt_key"))
    ctx = format_groups(r.get("groups", []))
    print(f"\nEXPECTED: {expected}")
    rows.append({"n": n, "question": q, "answer": r["answer"], "expected": expected, "route": gate(r),
                 "top_score": round(float(r.get("top_score", float("nan"))), 2),
                 "answer_from": "no model call (gate)" if g is None else "cache" if g["from_cache"] else "generated",
                 "think_tokens": g.get("think_tokens") if g else None,
                 "thinking_stopped": bool(g and g.get("thinking_forced")),
                 "answer_tokens": g.get("answer_tokens") if g else None, "cut_off": bool(g and g.get("hit_limit")),
                 "seconds": round(time.perf_counter() - t0, 1),
                 "figures_not_in_context": ", ".join(unsupported_figures(r["answer"], ctx, q)),
                 "sources": " | ".join(group_label(gr) for gr in r.get("groups", [])), "my_verdict": "",
                 "thinking": (g or {}).get("thinking", "")})

if rows:
    HARD_DF = pd.DataFrame(rows)
    _p = OUT / "hard_questions.csv"
    HARD_DF.to_csv(_p, index=False)
    mirror(_p)
    print(f"\n{'=' * 110}\n" + HARD_DF[["n", "route", "top_score", "answer_from", "think_tokens", "thinking_stopped",
                                         "answer_tokens", "cut_off", "seconds", "figures_not_in_context"]].to_string(index=False))
    print(f"\nSaved {_p}: answers next to the expected answers, with an empty 'my_verdict' column to fill in.")
else:
    print('Hard questions skipped (SETTINGS["hard_questions"] is False).')


In [ ]:
# ================================================================
# 9) Run: retrieval pass, gate check, answers (cached), evidence-removed test, repeat check
# ================================================================
_T0 = time.perf_counter()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

def load_run(src) -> dict:
    """A saved run: this notebook's <run>.json, a results zip, or a v2.6 compact run (its zip or json)."""
    p = upload_one("results file (.zip or .json)") if src == "upload" else Path(src)
    data = p.read_bytes()
    if p.suffix == ".zip":                     # the run is the JSON file that holds the per-question records
        z = zipfile.ZipFile(io.BytesIO(data))
        runs = [json.loads(z.read(n)) for n in z.namelist() if n.endswith(".json")]
        runs = [r for r in runs if isinstance(r, dict) and r.get("records")]
        assert runs, f"no run with question records in {p.name}"
        run = runs[0]
    else:
        run = json.loads(data)
    if "verdict" not in (run.get("records") or [{}])[0]:          # v2.6 compact evaluation format
        run = {"name": run.get("name", p.stem), "notebook_version": run.get("notebook_version"), "chunk_sha256": None,
               "models": (run.get("environment") or {}).get("models", {}), "testset_version": run.get("testset_version"),
               "records": [{"id": r["id"], "question": r["question"], "answer": r.get("answer"), "context": r.get("context", ""),
                            "gate_route": r.get("gate_route"), "top_score": r.get("top_score"), "gen": r.get("gen"),
                            "error": r.get("error"), "context_tokens": r.get("context_tokens"), "retrieval": {}}
                           for r in run["records"]]}
    return run

SRC = load_run(SETTINGS["contexts_from"]) if SETTINGS["contexts_from"] and SETTINGS["mode"] != "rescore" else None
RES = None
if SETTINGS["mode"] == "rescore":
    assert SETTINGS["rescore_from"], 'mode "rescore" needs SETTINGS["rescore_from"]: the saved run\'s zip or JSON'
    RES = load_run(SETTINGS["rescore_from"])
    print(f"Re-scoring the saved answers of run '{RES['name']}' ({len(RES['records'])} questions) with test set "
          f"{TESTSET_VERSION}; no answer model is used")
    if RES.get("chunk_sha256") != CHUNK_SHA:
        log("the saved run used a different chunk file: its retrieval metrics are kept as saved, not recomputed")
if SRC is not None:
    assert SETTINGS["mode"] == "full", "contexts_from re-answers saved contexts: use mode 'full'"
    print(f"Answering the saved contexts of run '{SRC['name']}' ({len(SRC['records'])} questions); retrieval is not repeated")

ENV = {"notebook_version": NOTEBOOK_VERSION, "started": datetime.datetime.now().isoformat(timespec="seconds"),
       "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
       "packages": {p: version(p) for p in ["torch", "transformers", "sentence-transformers", "bitsandbytes", "accelerate",
                                            "faiss-cpu", "rank-bm25", "langgraph"]},
       "models": MODELS, "gpu_after_loading_gb": gpu_gb(), "chunk_file": CHUNK_PATH.name, "chunk_sha256": CHUNK_SHA,
       "testset_version": TESTSET_VERSION, "splits": SETTINGS["splits"], "config": dict(CONFIG),
       "settings": {k: v for k, v in SETTINGS.items() if k != "overrides"}, "overrides": SETTINGS["overrides"]}

def _record(item: dict, state: dict) -> dict:
    cands = state.get("retrieved") or []
    scores = state.get("scores") or []
    ranked = sorted(([int(c["idx"]), round(float(s), 4)] for c, s in zip(cands, scores)), key=lambda x: (-x[1], x[0]))
    groups = [[int(i) for i in g] for g in state.get("groups", [])]
    ctx_ids = [i for g in groups for i in g]
    return {"id": item["id"], "split": item["split"], "type": item["type"], "answerable": item["answerable"],
            "paraphrase": item["paraphrase"], "question": item["question"], "premium_routed": bool(state.get("premium")),
            "candidates": [int(c["idx"]) for c in cands], "ranked": ranked, "groups": groups,
            "context": format_groups(groups), "context_tokens": sum(block_tokens(final_chunks[i]) for i in ctx_ids),
            "top_score": float(state["top_score"]) if np.isfinite(state.get("top_score", -np.inf)) else None,
            "gate_route": gate(state), "answer": None, "gen": None, "error": state.get("error"),
            "oom_retry": bool(state.get("oom_retry")), "retrieval": {}, "verdict": None, "grounding": None}

def retrieval_pass(items):
    recs = []
    for n, item in enumerate(items, 1):
        st = {"query": item["question"]}
        st.update(retrieve_node(st))
        st.update(rerank_node(st))
        recs.append(_record(item, st))
        if n % 25 == 0:
            print(f"  retrieval {n}/{len(items)}")
    return recs

def auroc(pos, neg):
    """P(score of an answerable question > score of a not-answerable one); ties count half."""
    if not pos or not neg:
        return None
    return float(np.mean([(p > q) + 0.5 * (p == q) for p in pos for q in neg]))

def gate_report(recs) -> dict:
    s = lambda r: r["top_score"] if r["top_score"] is not None else -99.0
    out = {}
    for split in sorted({r["split"] for r in recs}):
        rs = [r for r in recs if r["split"] == split]
        pos, neg = [s(r) for r in rs if r["answerable"]], [s(r) for r in rs if not r["answerable"]]
        thr = CONFIG["rerank_score_threshold"]
        out[split] = {"auroc": auroc(pos, neg), "answerable_refused_by_gate": sum(p < thr for p in pos),
                      "not_answerable_refused_by_gate": sum(q < thr for q in neg), "n_answerable": len(pos),
                      "n_not_answerable": len(neg), "lowest_answerable_score": round(min(pos), 2) if pos else None,
                      "highest_not_answerable_score": round(max(neg), 2) if neg else None}
    return out

# ---------------------------------------------------------------- 9.1 retrieval pass (no answer model)
if RES is not None:                                           # saved answers, retrieval and gate as they were
    _ids = {t["id"] for t in ITEMS}
    RECORDS = []
    for rec in RES["records"]:
        if rec["id"] in _ids:
            item = BY_ID[rec["id"]]
            rec.update({k: item[k] for k in ("split", "type", "answerable", "paraphrase")}, grounding=None)
            if RES.get("chunk_sha256") != CHUNK_SHA:
                rec["context_source"] = RES["name"]
            RECORDS.append(rec)
elif SRC is None:
    RECORDS = retrieval_pass(ITEMS)
else:
    _src = {r["id"]: r for r in SRC["records"]}
    RECORDS = []
    for item in ITEMS:
        r0 = _src.get(item["id"])
        if r0 is None:
            continue
        rec = {k: r0.get(k) for k in ("candidates", "ranked", "groups", "context", "context_tokens", "top_score",
                                      "gate_route", "premium_routed")}
        rec.update({"id": item["id"], "split": item["split"], "type": item["type"], "answerable": item["answerable"],
                    "paraphrase": item["paraphrase"], "question": item["question"], "answer": None, "gen": None,
                    "error": None, "oom_retry": False, "retrieval": r0.get("retrieval") or {}, "verdict": None,
                    "grounding": None, "context_source": SRC["name"]})
        RECORDS.append(rec)
GATE = gate_report(RECORDS)
for split, g in GATE.items():
    print(f"Gate [{split}]: AUROC {g['auroc'] if g['auroc'] is None else round(g['auroc'], 3)} | at threshold "
          f"{CONFIG['rerank_score_threshold']}: refuses {g['answerable_refused_by_gate']}/{g['n_answerable']} answerable and "
          f"{g['not_answerable_refused_by_gate']}/{g['n_not_answerable']} not-answerable questions | lowest answerable score "
          f"{g['lowest_answerable_score']}, highest not-answerable {g['highest_not_answerable_score']}")
_dev = [r for r in RECORDS if r["split"] == "dev" and r["top_score"] is not None]
if _dev and any(r["answerable"] for r in _dev) and SRC is None:
    _low = min(r["top_score"] for r in _dev if r["answerable"])
    _blk = sum(r["top_score"] < _low for r in _dev if not r["answerable"])
    print(f"  Data for choosing the threshold (dev only): at {_low:.2f}, the lowest score of an answerable dev question, the "
          f"gate would refuse {_blk}/{sum(not r['answerable'] for r in _dev)} not-answerable dev questions. A new threshold "
          "is an experiment: set it in overrides and compare runs.")

# ---------------------------------------------------------------- 9.2 answers
LLM_CALLS = {"generated": 0, "from_cache": 0}
RETRIEVAL_REPEAT = {"identical": 0, "different": []}
TOKENS_PER_SECOND = None
if SETTINGS["mode"] == "full":
    _prompts = [build_prompt(r["question"], r["context"]) for r in RECORDS if r["gate_route"] == "answer"]
    _need = sum(answer_key(p) not in CACHE for p in _prompts)
    if _need:
        t0 = time.perf_counter()
        _, _, _n = llm_generate(_prompts[0], 32, min_new_tokens=32, thinking=False)
        TOKENS_PER_SECOND = round(_n / max(time.perf_counter() - t0, 1e-6), 2)
    print(f"{len(_prompts)} of {len(RECORDS)} questions go to the answer model: {len(_prompts) - _need} cached, {_need} to generate"
          + (f" (about {_need * ((0.8 * CONFIG['think_budget'] if SETTINGS['enable_thinking'] else 0) + 300) / TOKENS_PER_SECOND / 60 + _need * 3 / 60:.0f}"
             f" min at {TOKENS_PER_SECOND} tokens/s)" if _need else ""))
    by_id = {r["id"]: r for r in RECORDS}
    for n, item in enumerate([BY_ID[r["id"]] for r in RECORDS], 1):
        rec = by_id[item["id"]]
        GEN_LOG.clear()
        if SRC is not None:                                     # saved context, same prompt template
            if rec["gate_route"] == "answer":
                rec["answer"] = generate_answer(build_prompt(item["question"], rec["context"]))
            else:
                rec["answer"] = NO_ANSWER_TEXT
        else:
            st = app.invoke({"query": item["question"]})
            again = _record(item, st)
            same = all(again[k] == rec[k] for k in ("candidates", "ranked", "groups", "gate_route"))
            if st.get("oom_retry"):
                rec.update({k: again[k] for k in ("groups", "context", "context_tokens")}, oom_retry=True)
            elif same:
                RETRIEVAL_REPEAT["identical"] += 1
            else:
                RETRIEVAL_REPEAT["different"].append(item["id"])
            rec["answer"], rec["error"] = st.get("answer"), st.get("error")
        g = next(iter(GEN_LOG.values()), None)
        if g:
            rec["gen"] = {k: g.get(k) for k in ("key", "prompt_tokens", "new_tokens", "think_tokens", "answer_tokens",
                                                "thinking_forced", "hit_limit", "seconds", "from_cache")}
            rec["thinking"] = g.get("thinking") or None
            LLM_CALLS["from_cache" if g["from_cache"] else "generated"] += 1
        if n % 10 == 0:
            print(f"  answered {n}/{len(RECORDS)} ({(time.perf_counter() - _T0) / 60:.1f} min)")
    if RETRIEVAL_REPEAT["different"]:
        log(f"retrieval differed between two passes for {RETRIEVAL_REPEAT['different']}")

# ---------------------------------------------------------------- 9.3 scores
def score_record(rec: dict, live: bool = True):
    item = BY_ID[rec["id"]]
    if live and rec.get("candidates") is not None and rec.get("context_source") is None:
        rec["retrieval"] = retrieval_metrics(item, rec["candidates"], [i for i, _ in rec["ranked"]],
                                             [i for g in rec["groups"] for i in g])
    if rec.get("answer") is not None:
        cand = norm_text(" ".join(final_chunks[i]["text"] for i in rec["candidates"])) \
            if live and rec.get("candidates") and rec.get("context_source") is None else None
        rec["verdict"] = score_answer(item, rec["answer"], rec["context"], rec.get("gen"), rec["gate_route"],
                                      rec.get("error"), cand)
    return rec

for rec in RECORDS:
    score_record(rec)

# ---------------------------------------------------------------- 9.4 evidence removed from the context
EVIDENCE_REMOVED = []
_ER_CANDIDATES = ["G03", "K02", "X03", "K05", "T01", "K03", "TY01"]
if SETTINGS["mode"] == "full" and SETTINGS["evidence_removed_test"]:
    if SRC is not None and SRC.get("chunk_sha256") != CHUNK_SHA:
        log("evidence-removed test skipped: the saved contexts come from a different chunk file")
    else:
        recs = {r["id"]: r for r in RECORDS}
        for qid in _ER_CANDIDATES:
            rec, item = recs.get(qid), BY_ID[qid]
            if len(EVIDENCE_REMOVED) >= 4 or not rec or rec["gate_route"] != "answer":
                continue
            holds = lambda i: any(anchor_in(e, final_chunks[i]["words"]) for e in item["evidence"]) or \
                any(fact_found(f, norm_text(final_chunks[i]["text"])) for f in item["facts"])
            keep = [[i for i in g if not holds(i)] for g in rec["groups"]]
            keep = [g for g in keep if g]
            ctx = format_groups(keep)
            removed = sum(map(len, rec["groups"])) - sum(map(len, keep))
            if not keep or not removed or any(fact_found(f, norm_text(ctx)) for f in item["facts"]):
                continue                     # evidence was never in the context, or cannot be taken out cleanly
            ans = generate_answer(build_prompt(item["question"], ctx))
            q_norm = norm_text(item["question"])
            stated = [fact_label(f) for f in item["facts"] if fact_found(f, norm_text(ans)) and not fact_found(f, q_norm)]
            unsup = unsupported_figures(ans, ctx, item["question"])
            ok = is_no_answer(ans) or (admits_gap(ans) and not stated and not unsup)
            EVIDENCE_REMOVED.append({"id": qid, "chunks_removed": removed,
                                     "pass": ok, "outcome": "refused" if is_no_answer(ans) else "said it is not shown" if ok
                                     else "stated facts from outside the context" if stated else "answered without the evidence",
                                     "facts_stated": stated, "figures_not_in_context": unsup, "answer": ans})
        print(f"Evidence removed: {sum(e['pass'] for e in EVIDENCE_REMOVED)}/{len(EVIDENCE_REMOVED)} answers said the "
              f"policy text does not show it | " + ", ".join(f"{e['id']}: {e['outcome']}" for e in EVIDENCE_REMOVED))

# ---------------------------------------------------------------- 9.5 repeat check (determinism)
REPEAT = None
if SETTINGS["mode"] == "full" and SETTINGS["repeat_check"]:
    _c = [r for r in RECORDS if r.get("gen") and r["gen"].get("new_tokens")]
    if _c:
        _r = min(_c, key=lambda r: r["gen"]["new_tokens"])
        _again = generate_answer(build_prompt(_r["question"], _r["context"]), use_cache=False)
        REPEAT = {"id": _r["id"], "identical": _again == _r["answer"]}
        print(f"Repeat check: {_r['id']} regenerated without the cache is {'identical' if REPEAT['identical'] else 'DIFFERENT'}")
        if not REPEAT["identical"]:
            log(f"regenerated answer to {_r['id']} differs: generation is not fully deterministic")

PEAK_GPU_GB = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None
if RES is not None:                                           # what the saved run measured, not this session
    PEAK_GPU_GB, TOKENS_PER_SECOND = RES.get("peak_gpu_gb"), RES.get("tokens_per_second")
    LLM_CALLS, RETRIEVAL_REPEAT = RES.get("llm_calls") or LLM_CALLS, RES.get("retrieval_repeat") or RETRIEVAL_REPEAT
    EVIDENCE_REMOVED, REPEAT = RES.get("evidence_removed") or [], RES.get("repeat")
    ENV.update(models=RES.get("models"), config=RES.get("config"), rescored_from=RES["name"],
               answers_started=RES.get("started"))
RUN = {"name": SETTINGS["run_name"], "notebook_version": NOTEBOOK_VERSION, "testset_version": TESTSET_VERSION,
       "chunk_file": CHUNK_PATH.name, "chunk_sha256": CHUNK_SHA, "models": MODELS, "config": dict(CONFIG),
       "settings": ENV["settings"], "overrides": SETTINGS["overrides"], "context_source": SRC["name"] if SRC else None,
       "started": ENV["started"], "gate": GATE, "llm_calls": LLM_CALLS, "retrieval_repeat": RETRIEVAL_REPEAT,
       "tokens_per_second": TOKENS_PER_SECOND, "peak_gpu_gb": PEAK_GPU_GB, "evidence_removed": EVIDENCE_REMOVED,
       "repeat": REPEAT, "records": RECORDS}
RUN["wall_seconds"] = round(time.perf_counter() - _T0, 1)
if RES is not None:
    RUN.update(models=RES.get("models"), config=RES.get("config"), settings=RES.get("settings"), started=RES.get("started"),
               wall_seconds=RES.get("wall_seconds"), rescored_from=RES["name"])
(OUT / f"{RUN['name']}.json").write_text(json.dumps(RUN, ensure_ascii=False, default=str), encoding="utf-8")
mirror(OUT / f"{RUN['name']}.json")
if SETTINGS["mode"] in ("full", "rescore"):
    _fc = [r["verdict"]["fully_correct"] for r in RECORDS if r.get("verdict")]
    print(f"Done in {RUN['wall_seconds'] / 60:.1f} min: {sum(_fc)}/{len(_fc)} fully correct | answers generated "
          f"{LLM_CALLS['generated']}, from cache {LLM_CALLS['from_cache']} | peak GPU {PEAK_GPU_GB} GB")
else:
    print(f"Retrieval run done in {RUN['wall_seconds'] / 60:.1f} min")


In [ ]:
# ================================================================
# 10) Groundedness: is each answer sentence supported by its context? (HHEM-2.1-Open)
# ================================================================
# Each claim sentence is checked against the context the model was given (score 0-1; supported at
# >= DECISION["hhem_threshold"]). Runs in fp32 (T5-based; fp16 overflows) after all answers exist, so it
# costs no answer-model calls. Meta sentences ("the policy text does not mention ...") are not claims.
_META = re.compile(r"\b(context|policy text|provided text|provided policy|the text|information provided|document|excerpt"
                   r"|provided sections?|given sections?)s?\b[^.]{0,80}\b(not|no)\b"
                   r"|\b(?:context|provided sections?|given sections?|policy sections)\b[^.]{0,40}\b(?:outlines?|includes?"
                   r"|discuss(?:es)?|address(?:es)?|focus(?:es)?|describes?)\b"
                   r"|\bnot (specified|mentioned|stated|provided|shown|given|listed|included) in\b|^not specified in the provided"
                   r" policy text|\bthere is no (?:mention|reference|provision|information)\b|\bno (?:mention|reference) of\b"
                   r"|\b(?:is|are) not (?:explicitly )?(?:mentioned|specified|stated|defined)\b|\bcannot be determined\b"
                   r"|\bdoes not (?:explicitly )?(?:mention|specify|state|address|describe|include)\b", re.I)
_ABBREV = re.compile(r"\b(Rs|No|Nos|Sr|vs|approx|e\.g|i\.e|etc|Dr|Ltd|Co|Excl|Sec)\.\s+", re.I)   # not sentence ends
HHEM = None

def claim_units(answer: str):
    units = []
    for line in (answer or "").splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)]|[a-zA-Z][.)]|#+)\s*", "", line).replace("**", "").strip()
        line = _ABBREV.sub(lambda m: m.group(1) + "\u2024 ", line)      # "Rs. 15,000" stays one sentence
        for s in re.split(r"(?<=[.!?])\s+(?=[A-Z(₹\d])", line):
            s = s.replace("\u2024", ".")
            s = s.strip()
            if len(re.findall(r"[A-Za-z]+", s)) < 4 or s.endswith(":") or _META.search(s) or "|" in s[:1]:
                continue
            units.append(s)
    return units

def ensure_hhem():
    global HHEM
    if HHEM is None:
        try:
            HHEM = AutoModelForSequenceClassification.from_pretrained(SETTINGS["grounding_model"], trust_remote_code=True,
                                                                      **dtype_kw(torch.float32)).to(DEVICE).eval()
        except Exception as e:
            log(f"grounding model could not be loaded ({type(e).__name__}: {e}); groundedness not measured")
            HHEM = False
    return HHEM

def hhem_scores(pairs, batch_size: int = 4):
    out = []
    for i in range(0, len(pairs), batch_size):
        part = pairs[i:i + batch_size]
        try:
            with torch.no_grad():
                out += [float(x) for x in HHEM.predict(part)]
        except torch.cuda.OutOfMemoryError:            # long context: one sentence at a time
            free_gpu()
            for pr in part:
                with torch.no_grad():
                    out.append(float(HHEM.predict([pr])[0]))
    return out

def ground_record(rec: dict):
    """Adds rec["grounding"]: share of supported claim sentences, and the weakest sentence."""
    a = rec.get("answer")
    if not a or is_no_answer(a) or not rec.get("context"):
        rec["grounding"] = None
        return rec
    units = claim_units(a)
    if not units:
        rec["grounding"] = {"units": 0, "supported": None, "groundedness": None, "hallucinated": False}
        return rec
    sc = hhem_scores([(rec["context"], u) for u in units])
    thr = DECISION["hhem_threshold"]
    worst = int(np.argmin(sc))
    rec["grounding"] = {"units": len(units), "supported": sum(s >= thr for s in sc),
                        "groundedness": sum(s >= thr for s in sc) / len(units), "hallucinated": any(s < thr for s in sc),
                        "weakest": units[worst][:200], "weakest_score": round(sc[worst], 3),
                        "unsupported": [u[:160] for u, s in zip(units, sc) if s < thr][:5]}
    return rec

def ground_run(run: dict, label: str = ""):
    if not ensure_hhem():
        return 0
    t0, n = time.perf_counter(), 0
    for rec in run["records"]:
        a = rec.get("answer")
        if a and not is_no_answer(a) and rec.get("context") and rec.get("grounding") is None:
            ground_record(rec)
            n += 1
    print(f"Groundedness{label}: {n} answers checked in {time.perf_counter() - t0:.0f}s")
    return n

if SETTINGS["mode"] in ("full", "rescore") and SETTINGS["groundedness"]:
    free_gpu()
    if ground_run(RUN):
        _g = [r["grounding"] for r in RUN["records"] if r.get("grounding") and r["grounding"]["groundedness"] is not None]
        if _g:
            print(f"  mean groundedness {np.mean([g['groundedness'] for g in _g]):.3f} | answers with an unsupported "
                  f"sentence: {sum(g['hallucinated'] for g in _g)}/{len(_g)}")
        RUN["grounding_model"] = SETTINGS["grounding_model"]
        RUN["grounding_revision"] = revision(HHEM, SETTINGS["grounding_model"])
        (OUT / f"{RUN['name']}.json").write_text(json.dumps(RUN, ensure_ascii=False, default=str), encoding="utf-8")
        mirror(OUT / f"{RUN['name']}.json")


In [ ]:
# ================================================================
# 11) Results: scorecard, failures by stage, report.md, one zip
# ================================================================
def wilson(k: int, n: int, z: float = 1.96):
    if not n:
        return (None, None)
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def mde(n: int, p_disc: float = 0.25, alpha: float = 0.05, power: float = 0.8):
    """Smallest difference in "fully correct" a paired McNemar test detects on n questions (80% power,
    5% two-sided), assuming a quarter of the questions change outcome between two systems."""
    za, zb = 1.959964, 0.841621
    for d in np.arange(0.01, 0.6, 0.005):
        if p_disc - d * d <= 0:
            return None
        if (za * math.sqrt(p_disc) + zb * math.sqrt(p_disc - d * d)) ** 2 / d ** 2 <= n:
            return round(float(d), 3)
    return None

_mean = lambda xs: float(np.mean(xs)) if len(xs) else None
_has = lambda r, kind: any(i["kind"] == kind for i in (r.get("verdict") or {}).get("issues", []))

def summarize(recs) -> dict:
    """All metrics for a set of question records (one split, or all)."""
    v = [r for r in recs if r.get("verdict")]
    ans = [r for r in v if r["answerable"]]
    neg = [r for r in v if not r["answerable"]]
    ret = [r["retrieval"] for r in recs if r["answerable"] and r.get("retrieval")]
    grd = [r["grounding"] for r in v if r.get("grounding") and r["grounding"].get("groundedness") is not None]
    tabs = [r["verdict"]["table"] for r in ans if r["verdict"].get("table")]
    fc = lambda rs: [bool(r["verdict"]["fully_correct"]) for r in rs]
    out = {"n": len(recs), "n_answerable": sum(r["answerable"] for r in recs)}
    if v:
        k = sum(fc(v))
        out.update(fully_correct=k / len(v), fully_correct_k=k, fully_correct_n=len(v), fully_correct_ci=wilson(k, len(v)),
                   fully_correct_answerable=_mean(fc(ans)), fully_correct_not_answerable=_mean(fc(neg)),
                   key_fact_recall=_mean([r["verdict"]["fact_recall"] or 0.0 for r in ans]),
                   critical_error_rate=_mean([bool(r["verdict"]["critical"]) for r in v]),
                   table_values=f"{sum(t['correct'] for t in tabs)}/{sum(t['expected'] for t in tabs)} correct, "
                                f"{sum(len(t['wrong']) for t in tabs)} wrong" if tabs else None,
                   list_completeness=_mean([r["verdict"]["fact_recall"] or 0.0 for r in ans if r["type"] == "list"]),
                   unsupported_figures=_mean([_has(r, "unsupported_figures") for r in v]),
                   unsupported_citations=_mean([_has(r, "unsupported_citations") for r in v]),
                   correct_abstention=_mean(fc(neg)),
                   false_refusal=_mean([bool(r["verdict"].get("false_refusal")) for r in ans]),
                   refused_by_gate=sum(r["gate_route"] == "no_answer" for r in ans),
                   everyday_gap=(_mean(fc([r for r in ans if not r["paraphrase"]])) or 0) -
                                (_mean(fc([r for r in ans if r["paraphrase"]])) or 0)
                   if any(r["paraphrase"] for r in ans) and any(not r["paraphrase"] for r in ans) else None,
                   cut_off=sum(_has(r, "cut_off") for r in v), errors=sum(bool(r.get("error")) for r in v),
                   oom_retries=sum(bool(r.get("oom_retry")) for r in v))
    if grd:
        out.update(groundedness=_mean([g["groundedness"] for g in grd]), hallucination_rate=_mean([g["hallucinated"] for g in grd]),
                   grounded_answers=len(grd))
    if ret:
        for m in ("context_recall", "search_recall", "mrr10", "context_precision", "evidence_in_chunk_file", "facts_in_context"):
            vals = [x[m] for x in ret if x.get(m) is not None]
            out[m] = _mean(vals)
    gens = [r["gen"] for r in recs if r.get("gen")]
    if gens:
        secs = [g["seconds"] for g in gens if g.get("seconds")]
        out.update(median_seconds=statistics.median(secs) if secs else None,
                   p95_seconds=float(np.percentile(secs, 95)) if secs else None,
                   median_prompt_tokens=statistics.median([g["prompt_tokens"] for g in gens if g.get("prompt_tokens")] or [0]),
                   median_answer_tokens=statistics.median([g.get("answer_tokens") or g.get("new_tokens") or 0 for g in gens]),
                   median_think_tokens=statistics.median([g.get("think_tokens") or 0 for g in gens])
                   if any(g.get("think_tokens") for g in gens) else None,
                   thinking_forced=sum(bool(g.get("thinking_forced")) for g in gens)
                   if any(g.get("think_tokens") for g in gens) else None)
    return out

def _fmt(x, pct=False):
    if x is None:
        return "–"
    if isinstance(x, str):
        return x
    if pct:
        return f"{100 * x:.1f}%"
    if isinstance(x, float):
        return f"{x:.0f}" if abs(x) >= 100 else f"{x:.1f}" if abs(x) >= 10 else f"{x:.3f}"
    return str(x)

SCORE_ROWS = [  # (group, label, key, as percent)
    ("PRIMARY", "Fully correct (all questions)", "fully_correct", True),
    ("Answer correctness", "fully correct: answerable", "fully_correct_answerable", True),
    ("Answer correctness", "key-fact recall (answerable)", "key_fact_recall", False),
    ("Answer correctness", "critical error rate", "critical_error_rate", True),
    ("Answer correctness", "plan-table values", "table_values", False),
    ("Answer correctness", "list completeness (list questions)", "list_completeness", False),
    ("Faithfulness", "groundedness (supported sentences, HHEM)", "groundedness", False),
    ("Faithfulness", "hallucination rate (answers with an unsupported sentence)", "hallucination_rate", True),
    ("Faithfulness", "answers with figures not in the context", "unsupported_figures", True),
    ("Faithfulness", "answers citing section numbers not in the context", "unsupported_citations", True),
    ("Retrieval", "context recall (evidence passages reaching the model)", "context_recall", False),
    ("Retrieval", "search recall (evidence among the candidates)", "search_recall", False),
    ("Retrieval", "MRR@10 (rank of the first relevant chunk)", "mrr10", False),
    ("Retrieval", "context precision (share of context tokens that are evidence)", "context_precision", False),
    ("Retrieval", "evidence present in the chunk file", "evidence_in_chunk_file", False),
    ("Abstention", "correct abstention (not answerable)", "correct_abstention", True),
    ("Abstention", "false refusal rate (answerable)", "false_refusal", True),
    ("Abstention", "answerable questions refused by the gate", "refused_by_gate", False),
    ("Robustness", "everyday-wording gap (policy wording minus everyday wording)", "everyday_gap", True),
    ("Efficiency", "seconds per answer, median", "median_seconds", False),
    ("Efficiency", "seconds per answer, 95th percentile", "p95_seconds", False),
    ("Efficiency", "prompt tokens, median", "median_prompt_tokens", False),
    ("Efficiency", "answer tokens, median", "median_answer_tokens", False),
    ("Efficiency", "thinking tokens, median", "median_think_tokens", False),
    ("Reliability", "thinking stopped at the budget (told to answer)", "thinking_forced", False),
    ("Reliability", "answers cut off at the length limit", "cut_off", False),
    ("Reliability", "pipeline errors / out-of-memory retries", "errors", False),
]

def scorecard(run: dict) -> pd.DataFrame:
    recs = run["records"]
    groups = {s: summarize([r for r in recs if r["split"] == s]) for s in ("dev", "test") if any(r["split"] == s for r in recs)}
    if len(groups) > 1:
        groups["all"] = summarize(recs)
    rows = []
    for grp, label, key, pct in SCORE_ROWS:
        vals = {s: g.get(key) for s, g in groups.items()}
        if all(x is None for x in vals.values()):
            continue
        row = {"group": grp, "measure": label}
        for s, g in groups.items():
            x = g.get(key)
            if key == "fully_correct" and x is not None:
                lo, hi = g["fully_correct_ci"]
                row[s] = f"{g['fully_correct_k']}/{g['fully_correct_n']} = {100 * x:.1f}% (95% CI {100 * lo:.0f}–{100 * hi:.0f}%)"
            elif key == "errors" and x is not None:
                row[s] = f"{x} / {g['oom_retries']}"
            else:
                row[s] = _fmt(x, pct)
        rows.append(row)
    answered = any(r.get("gen") for r in recs)
    for grp, label, val in [
            ("Abstention", "gate separability (AUROC of the best reranker score)",
             {s: _fmt((run.get("gate") or {}).get(s, {}).get("auroc")) for s in groups if s != "all"}),
            ("Abstention", "evidence removed: answer says it is not shown",
             {"dev": f"{sum(e['pass'] for e in run.get('evidence_removed') or [])}/{len(run.get('evidence_removed') or [])}"}
             if run.get("evidence_removed") else {}),
            ("Efficiency", "tokens per second / peak GPU memory (GB)",
             {"all" if "all" in groups else next(iter(groups)): f"{run.get('tokens_per_second') or '–'} / {run.get('peak_gpu_gb') or '–'}"}
             if answered else {}),
            ("Reliability", "retrieval identical when repeated / regenerated answer identical",
             {"all" if "all" in groups else next(iter(groups)):
              f"{run['retrieval_repeat']['identical']}/{run['retrieval_repeat']['identical'] + len(run['retrieval_repeat']['different'])}"
              f" / {(run.get('repeat') or {}).get('identical', 'not run')}"} if answered and run.get("retrieval_repeat") else {})]:
        if val:
            rows.append({"group": grp, "measure": label, **val})
    return pd.DataFrame(rows).fillna("")

def question_table(run: dict) -> pd.DataFrame:
    rows = []
    for r in run["records"]:
        v, rt, g, gen = r.get("verdict") or {}, r.get("retrieval") or {}, r.get("grounding") or {}, r.get("gen") or {}
        item = BY_ID.get(r["id"], {})
        rows.append({"id": r["id"], "split": r["split"], "type": r["type"], "answerable": r["answerable"],
                     "fully_correct": v.get("fully_correct"), "first_stage": v.get("first_stage") or "",
                     "critical": ", ".join(v.get("critical", [])), "fact_recall": v.get("fact_recall"),
                     "context_recall": rt.get("context_recall"), "mrr10": rt.get("mrr10"),
                     "context_precision": None if rt.get("context_precision") is None else round(rt["context_precision"], 2),
                     "groundedness": None if g.get("groundedness") is None else round(g["groundedness"], 2),
                     "gate": r["gate_route"], "top_score": r.get("top_score"), "context_tokens": r.get("context_tokens"),
                     "answer_tokens": gen.get("answer_tokens") or gen.get("new_tokens"), "think_tokens": gen.get("think_tokens"),
                     "thinking_forced": gen.get("thinking_forced"), "seconds": gen.get("seconds"),
                     "issues": " | ".join(i["issue"] for i in v.get("issues", [])),
                     "weakest_sentence": g.get("weakest", ""), "question": r["question"], "answer": r.get("answer"),
                     "reference": item.get("reference", ""), "thinking": r.get("thinking") or ""})
    return pd.DataFrame(rows)

def stage_table(run: dict) -> pd.DataFrame:
    rows = [{"stage": i["stage"], "kind": i["kind"], "id": r["id"]} for r in run["records"]
            for i in (r.get("verdict") or {}).get("issues", [])]
    if not rows:
        return pd.DataFrame(columns=["stage", "kind", "questions"])
    t = pd.DataFrame(rows).drop_duplicates().groupby(["stage", "kind"])["id"].apply(lambda s: ", ".join(sorted(s))).reset_index(name="questions")
    t["stage"] = pd.Categorical(t["stage"], STAGES, ordered=True)
    return t.sort_values(["stage", "kind"]).reset_index(drop=True)

def _cell(x) -> str:
    if isinstance(x, float):
        if math.isnan(x):
            return ""
        return str(int(x)) if x.is_integer() else f"{x:.3f}"
    return str(x).replace("|", "/").replace("\n", " ")

def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(_cell(row[c]) for c in cols) + " |")
    return "\n".join(out)

SCORECARD = scorecard(RUN)
QUESTIONS = question_table(RUN)
STAGE_TABLE = stage_table(RUN)
print("=== Scorecard (decisions use the held-out test split)")
print(SCORECARD.to_string(index=False))
if SETTINGS["mode"] in ("full", "rescore"):
    print("\n=== First failing stage per question")
    print(QUESTIONS.query("fully_correct == False")["first_stage"].value_counts().to_string())
    print("\n=== Issues by stage")
    print(STAGE_TABLE.to_string(index=False))
print("\n=== Questions")
print(QUESTIONS[["id", "split", "type", "fully_correct", "first_stage", "fact_recall", "context_recall", "mrr10",
                 "groundedness", "issues"] if SETTINGS["mode"] != "retrieval" else
                ["id", "split", "type", "answerable", "context_recall", "mrr10", "context_precision", "top_score",
                 "context_tokens"]].to_string(index=False, max_colwidth=90))

_n = {s: sum(r["split"] == s for r in RUN["records"]) for s in ("dev", "test")}
L = [f"# Evaluation: {RUN['name']}", "",
     f"Notebook {NOTEBOOK_VERSION} | test set {TESTSET_VERSION} | {ENV['started']} | GPU {ENV['gpu']} | chunk file "
     f"{CHUNK_PATH.name} (sha256 {CHUNK_SHA[:12]}) | answer model {RUN['models']['llm']} ({RUN['models']['llm_precision']}, "
     f"thinking {'on' if RUN['models']['enable_thinking'] else 'off'}) | reranker {RUN['models']['reranker']} | embedder "
     f"{RUN['models']['embed']}" + (f" | answers from run '{RUN['rescored_from']}' ({RUN['started']}), re-scored"
                                    if RUN.get("rescored_from") else "")
     + (f" | contexts from run '{RUN['context_source']}'" if RUN.get("context_source") else "")
     + (f" | overrides {SETTINGS['overrides']}" if SETTINGS["overrides"] else ""), "",
     "## Scorecard", "", md_table(SCORECARD), "",
     "Reading it: the unit is the question. Smallest difference in *fully correct* a paired test can detect "
     f"(80% power): about {100 * (mde(_n['test']) or 0):.0f} points on the {_n['test']} test questions, about "
     f"{100 * (mde(_n['dev']) or 0):.0f} on the {_n['dev']} dev questions. Dev results guide debugging; adopt or "
     "reject only on test. Checks are rules on text; `results.csv` has every answer next to its reference.", ""]
if SETTINGS["mode"] in ("full", "rescore"):
    L += ["## Where answers go wrong (first failing stage)", "",
          md_table(QUESTIONS.query("fully_correct == False")["first_stage"].value_counts().rename_axis("stage").reset_index(name="questions")),
          "", "## Issues by stage", "", md_table(STAGE_TABLE) if len(STAGE_TABLE) else "None.", ""]
L += ["## Gate", "", md_table(pd.DataFrame(RUN["gate"]).T.reset_index(names="split")), "",
      "## Questions", "", md_table(QUESTIONS[["id", "split", "type", "fully_correct", "first_stage", "critical", "fact_recall",
                                             "context_recall", "mrr10", "context_precision", "groundedness", "context_tokens",
                                             "think_tokens", "answer_tokens", "issues"]]), ""]
if RUN.get("evidence_removed"):
    L += ["## Evidence removed from the context", "",
          md_table(pd.DataFrame(RUN["evidence_removed"])[["id", "chunks_removed", "outcome", "facts_stated", "figures_not_in_context"]]), ""]
L += ["## Problems noted during the run", ""] + ([f"- {m}" for m in RUN_LOG] or ["None."]) + \
     ["", "## Test set against this chunk file", "",
      md_table(TESTSET_PROBLEMS) if len(TESTSET_PROBLEMS) else "All facts and evidence found; all out-of-scope terms absent."]
(OUT / "report.md").write_text("\n".join(L), encoding="utf-8")
QUESTIONS.to_csv(OUT / "results.csv", index=False)
SCORECARD.to_csv(OUT / "scorecard.csv", index=False)
ENV.update(finished=datetime.datetime.now().isoformat(timespec="seconds"), peak_gpu_gb=RUN["peak_gpu_gb"],
           tokens_per_second=RUN["tokens_per_second"], minutes=round((time.perf_counter() - _T0) / 60, 1), run_log=RUN_LOG)
(OUT / "environment.json").write_text(json.dumps(ENV, indent=1, ensure_ascii=False, default=str), encoding="utf-8")

def make_zip() -> Path:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    z = Path(f"rag_eval_{RUN['name']}_{stamp}.zip")
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(OUT.iterdir()):
            if p.is_file():
                f.write(p, f"{z.stem}/{p.name}")
    mirror(z)
    return z

ZIP = make_zip()
print(f"\nSaved {ZIP} ({ZIP.stat().st_size / 1024:.0f} KB): report.md, results.csv, scorecard.csv, {RUN['name']}.json, "
      "answer_cache.jsonl, environment.json (downloaded after cell 13). It contains policy text: keep it out of public repositories.")


## Comparison and validation of the checks

In [ ]:
# ================================================================
# 12) Compare with earlier runs: paired tests and the decision rule
# ================================================================
# SETTINGS["compare_with"] = the baseline run (zip/json), or [baseline, other candidate runs...]. This run is a
# candidate. Every run is re-scored with this notebook's checks first, so all are judged by the same rules.
# Primary: "fully correct", McNemar exact test on the questions both runs answered (Holm-adjusted when there
# are several candidates). Secondary: paired bootstrap 95% intervals. Decisions only on the test split.
def rescore(run: dict) -> dict:
    same_corpus = run.get("chunk_sha256") == CHUNK_SHA
    for rec in run["records"]:
        if rec["id"] in BY_ID:
            item = BY_ID[rec["id"]]
            rec.update({k: rec.get(k, item[k]) for k in ("type", "answerable", "paraphrase")})
            rec["split"] = item["split"]
            if same_corpus and rec.get("candidates") is not None and rec.get("context_source") is None:
                score_record(rec, live=True)
            elif rec.get("answer") is not None:
                rec["verdict"] = score_answer(item, rec["answer"], rec.get("context", ""), rec.get("gen"),
                                              rec.get("gate_route") or "answer", rec.get("error"))
    run["records"] = [r for r in run["records"] if r["id"] in BY_ID]
    return run

def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value (binomial test on the discordant questions)."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

def holm(pvals: dict) -> dict:
    order = sorted(pvals, key=pvals.get)
    out, running = {}, 0.0
    for rank, k in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * pvals[k]))
        out[k] = running
    return out

def per_question(run: dict) -> pd.DataFrame:
    rows = []
    for r in run["records"]:
        v, g, rt = r.get("verdict") or {}, r.get("grounding") or {}, r.get("retrieval") or {}
        rows.append({"id": r["id"], "split": r["split"], "answerable": r["answerable"],
                     "fully_correct": None if not v else float(bool(v["fully_correct"])),
                     "key_fact_recall": (v.get("fact_recall") or 0.0) if v and r["answerable"] else None,
                     "critical_error": None if not v else float(bool(v["critical"])),
                     "correct_abstention": None if not v or r["answerable"] else float(bool(v["fully_correct"])),
                     "groundedness": g.get("groundedness"), "hallucinated": None if not g else float(bool(g.get("hallucinated"))),
                     "context_recall": rt.get("context_recall"), "mrr10": rt.get("mrr10")})
    return pd.DataFrame(rows).set_index("id")

SECONDARY = [("key_fact_recall", 1), ("critical_error", -1), ("groundedness", 1), ("hallucinated", -1),
             ("correct_abstention", 1), ("context_recall", 1), ("mrr10", 1)]

def compare(base: dict, cand: dict, split: str, seed: int = 0) -> dict:
    a, b = per_question(base), per_question(cand)
    a, b = a[a.split == split], b[b.split == split]
    ids = a.index.intersection(b.index)
    rng = np.random.default_rng(seed)
    out = {"split": split, "n": len(ids), "rows": []}
    pa, pb = a.loc[ids, "fully_correct"], b.loc[ids, "fully_correct"]
    ok = pa.notna() & pb.notna()
    pa, pb = pa[ok], pb[ok]
    if len(pa):
        bb, cc = int(((pa == 1) & (pb == 0)).sum()), int(((pa == 0) & (pb == 1)).sum())
        d = (pb - pa).to_numpy()
        boots = d[rng.integers(0, len(d), (DECISION["n_bootstrap"], len(d)))].mean(axis=1)
        out.update(primary={"base": pa.mean(), "cand": pb.mean(), "diff": d.mean(), "ci": tuple(np.percentile(boots, [2.5, 97.5])),
                            "fixed": cc, "broken": bb, "p": mcnemar_exact(bb, cc), "n": len(d)},
                   fixed_ids=list(pa.index[(pa == 0) & (pb == 1)]), broken_ids=list(pa.index[(pa == 1) & (pb == 0)]))
    for m, sign in SECONDARY:
        x, y = a.loc[ids, m], b.loc[ids, m]
        k = x.notna() & y.notna()
        if not k.any():
            continue
        d = (y[k] - x[k]).to_numpy(float)
        boots = d[rng.integers(0, len(d), (DECISION["n_bootstrap"], len(d)))].mean(axis=1)
        lo, hi = np.percentile(boots, [2.5, 97.5])
        g_lo, g_hi = sorted((sign * lo, sign * hi))
        out["rows"].append({"metric": m, "n": int(k.sum()), "base": round(x[k].mean(), 3), "candidate": round(y[k].mean(), 3),
                            "diff": round(d.mean(), 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3),
                            "verdict": "better" if g_lo > 0 else "worse" if g_hi < 0 else "no clear change"})
    return out

def run_cost(run: dict) -> dict:
    s = summarize(run["records"])
    return {"median_seconds": s.get("median_seconds"), "peak_gpu_gb": run.get("peak_gpu_gb")}

def decide(res: dict, p_adj: float, cost: dict) -> str:
    if res["split"] != "test":
        return "dev only: guides debugging, no decision"
    pr = res.get("primary")
    if not pr:
        return "no paired answers"
    worse = {r["metric"] for r in res["rows"] if r["verdict"] == "worse"}
    reasons = []
    if not (p_adj < DECISION["alpha"] and pr["diff"] > 0):
        reasons.append(f"fully correct not significantly better (p = {p_adj:.3f})")
    for m in ("groundedness", "hallucinated", "correct_abstention"):
        if m in worse:
            reasons.append(f"{m} significantly worse")
    if cost["median_seconds"] and cost["median_seconds"] > DECISION["max_median_seconds"]:
        reasons.append(f"median {cost['median_seconds']:.0f}s per answer > {DECISION['max_median_seconds']}s")
    if cost["peak_gpu_gb"] and cost["peak_gpu_gb"] > DECISION["max_peak_gpu_gb"]:
        reasons.append(f"peak GPU {cost['peak_gpu_gb']} GB > {DECISION['max_peak_gpu_gb']} GB")
    return "ADOPT" if not reasons else "KEEP THE BASELINE: " + "; ".join(reasons)

def blind_sheet(base: dict, cand: dict, ids=None, seed: int = 7) -> Path:
    """Answers of two runs side by side as A/B in random order per question (model names hidden) for a human
    read; the key is saved separately (blind_key.json)."""
    rng, a, b = random.Random(seed), {r["id"]: r for r in base["records"]}, {r["id"]: r for r in cand["records"]}
    rows, key = [], {}
    for qid in ids or [i for i in b if i in a and a[i].get("answer") and b[i].get("answer") and a[i]["answer"] != b[i]["answer"]]:
        flip = rng.random() < 0.5
        x, y = (b[qid], a[qid]) if flip else (a[qid], b[qid])
        key[qid] = {"A": (cand if flip else base)["name"], "B": (base if flip else cand)["name"]}
        rows.append({"id": qid, "question": BY_ID[qid]["question"], "reference": BY_ID[qid]["reference"],
                     "answer_A": x["answer"], "answer_B": y["answer"], "better (A/B/same)": "", "notes": ""})
    p = OUT / "blind_review.csv"
    pd.DataFrame(rows).to_csv(p, index=False)
    (OUT / "blind_key.json").write_text(json.dumps(key, indent=1))
    return p

COMPARISONS = []
if SETTINGS["compare_with"]:
    _srcs = SETTINGS["compare_with"] if isinstance(SETTINGS["compare_with"], list) else [SETTINGS["compare_with"]]
    _runs = [rescore(load_run(s)) for s in _srcs]
    BASE, CANDS = _runs[0], [RUN] + _runs[1:]
    if SETTINGS["groundedness"] and any(r.get("answer") and r.get("grounding") is None and r.get("context")
                                        for r in BASE["records"]):
        ground_run(BASE, f" (baseline '{BASE['name']}')")
    for split in [s for s in ("dev", "test") if s in SETTINGS["splits"]]:
        res = {c["name"]: compare(BASE, c, split) for c in CANDS}
        p_adj = holm({k: r["primary"]["p"] for k, r in res.items() if r.get("primary")})
        for c in CANDS:
            r = res[c["name"]]
            if not r.get("primary"):
                continue
            pr, verdict = r["primary"], decide(r, p_adj[c["name"]], run_cost(c))
            COMPARISONS.append({"split": split, "baseline": BASE["name"], "candidate": c["name"], "n": pr["n"],
                                "fully_correct_base": round(pr["base"], 3), "fully_correct_candidate": round(pr["cand"], 3),
                                "diff": round(pr["diff"], 3), "ci": f"{pr['ci'][0]:+.3f} to {pr['ci'][1]:+.3f}",
                                "fixed": pr["fixed"], "broken": pr["broken"], "p_mcnemar": round(pr["p"], 4),
                                "p_holm": round(p_adj[c["name"]], 4), "decision": verdict})
            print(f"\n=== [{split}] {BASE['name']} -> {c['name']}: fully correct {pr['base']:.3f} -> {pr['cand']:.3f} "
                  f"(diff {pr['diff']:+.3f}, 95% CI {pr['ci'][0]:+.3f} to {pr['ci'][1]:+.3f}; fixed {pr['fixed']}, broken "
                  f"{pr['broken']}; McNemar p {pr['p']:.4f}, Holm {p_adj[c['name']]:.4f}) | {verdict}")
            print(f"    now correct: {r['fixed_ids'] or 'none'} | now wrong: {r['broken_ids'] or 'none'}")
            print(pd.DataFrame(r["rows"]).to_string(index=False))
    if COMPARISONS:
        pd.DataFrame(COMPARISONS).to_csv(OUT / "comparison.csv", index=False)
        _bs = blind_sheet(BASE, RUN)
        ZIP = make_zip()
        print(f"\nSaved comparison.csv and {_bs.name} (answers that differ, as A/B with names hidden; key in "
              f"blind_key.json) in {ZIP}")
else:
    print("No comparison: this run is the baseline. Later, set SETTINGS['compare_with'] to this run's zip.")



In [ ]:
# ================================================================
# 13) Check the checks: does the automatic verdict agree with a human reading? (Cohen's kappa)
# ================================================================
# 1. This cell saves labels_to_fill.csv: 30 answers, stratified by verdict and question type, WITHOUT the
#    automatic verdict. Mark human_fully_correct as 1 or 0 (all key facts, nothing false or missing).
# 2. Set LABELS_FILE to the filled file (or "upload") and run this cell again: kappa and every disagreement.
#    Target: kappa >= DECISION["kappa_target"]; fix a check that disagrees for a clear reason.
LABELS_FILE = None

# The human reading of the 24 answers of the v2.6 compact baseline run ("compact_baseline", Mistral-7B 4-bit,
# 20 Oct 2026): used automatically when that run is in SETTINGS["compare_with"]. N05 is left out: it became
# answerable in test set v3 (the PDF has the premium tables).
B0_READING = {"G09": 0, "C01": 0, "T01": 1, "G08": 0, "G07": 0, "G02": 0, "G03": 0, "G04": 0, "G05": 1, "G06": 1,
              "C03": 1, "M01": 0, "K03": 1, "K07": 0, "K05": 1, "K02": 1, "X03": 0, "TY01": 1, "PA01": 1, "FP03": 0,
              "FP01": 1, "N04": 0, "N01": 1}

def cohen_kappa(x, y):
    x, y = np.asarray(x, int), np.asarray(y, int)
    po = float((x == y).mean())
    pe = float(x.mean() * y.mean() + (1 - x.mean()) * (1 - y.mean()))
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)

def agreement(run: dict, labels: dict, title: str):
    recs = {r["id"]: r for r in run["records"] if r.get("verdict")}
    ids = [i for i in labels if i in recs]
    if not ids:
        print(f"{title}: no labelled answers in this run")
        return None
    auto = [int(bool(recs[i]["verdict"]["fully_correct"])) for i in ids]
    human = [int(labels[i]) for i in ids]
    k = cohen_kappa(auto, human)
    dis = pd.DataFrame([{"id": i, "human": h, "automatic": a, "issues": " | ".join(x["issue"] for x in recs[i]["verdict"]["issues"])}
                        for i, h, a in zip(ids, human, auto) if h != a])
    print(f"{title}: kappa {k:.2f} on {len(ids)} answers (agree on {len(ids) - len(dis)}); target >= {DECISION['kappa_target']}")
    if len(dis):
        print(dis.to_string(index=False))
    return {"kappa": k, "n": len(ids), "disagreements": dis.to_dict("records")}

def label_sheet(run: dict, n: int = 30, seed: int = 11) -> Path:
    rng = random.Random(seed)
    recs = [r for r in run["records"] if r.get("verdict")]
    strata = {}
    for r in recs:
        strata.setdefault((bool(r["verdict"]["fully_correct"]), r["answerable"]), []).append(r)
    picked = []
    for key in sorted(strata, key=lambda k: -len(strata[k])):
        pool = sorted(strata[key], key=lambda r: (r["type"], r["id"]))
        share = max(2, round(n * len(pool) / max(1, len(recs))))
        picked += rng.sample(pool, min(share, len(pool)))
    picked = sorted(picked[:n], key=lambda r: r["id"])
    p = OUT / "labels_to_fill.csv"
    pd.DataFrame([{"id": r["id"], "split": r["split"], "type": r["type"], "question": r["question"],
                   "reference": BY_ID[r["id"]]["reference"], "answer": r["answer"], "human_fully_correct": "", "notes": ""}
                  for r in picked]).to_csv(p, index=False)
    return p

VALIDATION = {}
if SETTINGS["mode"] in ("full", "rescore"):
    _sheet = label_sheet(RUN)
    print(f"Saved {_sheet} ({len(pd.read_csv(_sheet))} answers to label)")
    if LABELS_FILE:
        _lab = pd.read_csv(upload_one("filled labels CSV") if LABELS_FILE == "upload" else LABELS_FILE)
        _lab = _lab[pd.to_numeric(_lab["human_fully_correct"], errors="coerce").notna()]
        VALIDATION["this_run"] = agreement(RUN, dict(zip(_lab["id"], _lab["human_fully_correct"].astype(int))), "This run")
for _r in [BASE] if SETTINGS["compare_with"] else []:
    if _r["name"] == "compact_baseline":
        VALIDATION["compact_baseline"] = agreement(_r, B0_READING, "Checks vs the human reading of the v2.6 baseline answers")
if VALIDATION:
    (OUT / "validation.json").write_text(json.dumps(VALIDATION, indent=1, default=str))
ZIP = make_zip()
print(f"Final results: {ZIP}")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))
